# Project Fertility PSP

## 난임 환자 대상 임신 성공 여부 예측 프로젝트

본 프로젝트는 난임 환자 치료 데이터를 활용하여, 임신 성공 여부를 예측하는 분류 모델을 개발하는 과정이다.

분석은 데이터 탐색(EDA), 전처리 및 파생변수 생성, 모델링, 평가 순으로 진행한다. 

특히 본 프로젝트에서는 결측값을 단순 누락으로 보지 않고, 시술 프로세스와 연결된 신호로 해석하는 방향을 함께 고려한다.

## 00. 프로젝트 개요 및 분석 전략

본 프로젝트의 핵심 목표는 테스트 데이터에 대해 임신 성공 확률을 예측하는 것이다. 
평가지표는 ROC-AUC를 중심으로 하며, 예측 확률이 성공 사례와 실패 사례를 얼마나 잘 구분하는지에 중점을 둔다.

분석 전략은 다음과 같다.

- 결측값을 단순 제거하지 않고 시술 과정의 신호로 해석한다.
- 나이, 난자 수, 배아 수 등 주요 변수는 임상적 의미를 고려해 구간화한다.
- 범주형 변수와 조합 변수를 적극적으로 활용한다.
- CatBoost, LightGBM, XGBoost 등 트리 기반 모델을 비교한다.
- 최종 단계에서는 K-Fold OOF 검증과 앙상블을 통해 성능을 개선한다.

## 목차

01. 데이터 탐색(EDA)
- 데이터 로드 및 기본 구조 확인
- 타깃 분포 및 클래스 불균형 확인
- 결측치 현황 확인
- 시술 유형별 구조적 결측 확인
- 주요 변수별 임신 성공률 확인
- 나이, 난자, 배아 관련 구간화 기준 탐색

02. 데이터 전처리 및 Feature Engineering
- 기본 전처리 방향
- 임상 지식 기반 결측 유추
- 구조적 결측 flag 생성
- 구간화 변수 생성
- 조합 변수 및 비율 변수 생성
- 인코딩 및 데이터 분리

03. 모델링
- Baseline 모델
- CatBoost 모델
- LightGBM / XGBoost 모델
- K-Fold OOF 학습
- 앙상블 전략

04. 평가
- ROC-AUC 중심 평가
- 보조 지표 확인
- Feature Importance 및 Noise Pruning

05. 제출 및 리더보드 확인
- 제출 파일 생성
- 제출 파일 검증
- 리더보드 결과 기록


## 01. 데이터 탐색(EDA)

EDA 단계에서는 데이터의 기본 구조를 확인하고, 이후 전처리 및 파생변수 생성으로 연결될 수 있는 신호를 찾는다. 단순히 표와 그래프를 확인하는 것이 아니라, 모델이 학습할 수 있는 구조로 어떻게 변환할지까지 함께 고려한다.


### 01-1. 데이터 로드 및 기본 구조 확인

먼저 분석에 필요한 라이브러리와 시각화 테마를 설정하고, train/test/submission 파일을 불러온다.

이 단계에서 확인할 내용은 다음과 같다.

- train 데이터의 행/열 개수
- test 데이터의 행/열 개수
- 제출 파일 형식
- target 컬럼 존재 여부
- train/test 컬럼 차이


In [ ]:
# 라이브러리 및 시각화 기본 세팅

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
from IPython.display import display

warnings.filterwarnings("ignore")

# 경로 설정
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR / "outputs" / "project_fertility_psp"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 주요 컬럼 설정
TARGET = "임신 성공 여부"
ID_COL = "ID"
RANDOM_STATE = 42

# 한글 폰트 설정
font_candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic"]
available_fonts = {f.name for f in fm.fontManager.ttflist}

for font in font_candidates:
    if font in available_fonts:
        plt.rcParams["font.family"] = font
        break

plt.rcParams["axes.unicode_minus"] = False

# 모던한 다크 테마
sns.set_theme(style="darkgrid")

plt.rcParams.update({
    "figure.facecolor": "#111111",
    "axes.facecolor": "#111111",
    "axes.edgecolor": "#444444",
    "axes.labelcolor": "#F2F2F2",
    "xtick.color": "#EAEAEA",
    "ytick.color": "#EAEAEA",
    "text.color": "#F2F2F2",
    "axes.titlecolor": "#FFFFFF",
    "grid.color": "#333333",
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 11,
})

# seaborn 테마 적용 후 한글 폰트를 다시 고정
for font in font_candidates:
    if font in available_fonts:
        plt.rcParams["font.family"] = font
        break

main_color = "#6EA8FE"
accent_color = "#FFB86B"
sub_color = "#8BE9A6"

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("font:", plt.rcParams["font.family"])


In [ ]:
# 데이터 로드

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train.shape)
print("test:", test.shape)
print("submission:", submission.shape)

assert TARGET in train.columns, f"target 컬럼을 찾을 수 없습니다: {TARGET}"
assert ID_COL in train.columns, f"ID 컬럼을 찾을 수 없습니다: {ID_COL}"
assert ID_COL in test.columns, f"test 데이터에서 ID 컬럼을 찾을 수 없습니다: {ID_COL}"

In [ ]:
# 데이터 미리보기

display(train.head())
display(test.head())
display(submission.head())

In [ ]:
# train/test 컬럼 구조 확인

train_cols = set(train.columns)
test_cols = set(test.columns)

print("train columns:", train.shape[1])
print("test columns:", test.shape[1])
print("train에만 있는 컬럼:", sorted(train_cols - test_cols))
print("test에만 있는 컬럼:", sorted(test_cols - train_cols))

for i, col in enumerate(train.columns):
    print(i, col)


#### 해석 및 불균형 확인 필요성

train 데이터는 69개 컬럼, test 데이터는 68개 컬럼으로 구성되어 있다.  
train의 `임신 성공 여부`는 본 프로젝트의 예측 대상인 target 변수이다.

컬럼은 크게 다음과 같은 정보로 나눌 수 있다.

- 시술 기본 정보: `시술 시기 코드`, `시술 당시 나이`, `시술 유형`, `특정 시술 유형`
- 배란 및 유전검사 정보: `배란 자극 여부`, `배란 유도 유형`, `PGD/PGS 시술 여부`
- 불임 원인 정보: 남성/여성/부부 관련 불임 원인 변수
- 시술 이력 정보: 총 시술 횟수, IVF/DI 시술·임신·출산 횟수
- 난자/배아/정자 관련 정보: 생성 배아 수, 이식 배아 수, 난자 출처, 정자 출처 등
- 경과일 정보: 난자 채취, 난자 해동, 난자 혼합, 배아 이식, 배아 해동 경과일

다음 단계에서는 target 변수인 `임신 성공 여부`의 분포를 확인하여 클래스 불균형 여부를 살펴본다.

### 01-2. 타깃 분포 및 클래스 불균형 확인

타깃 변수인 `임신 성공 여부`의 0/1 분포를 확인한다. 본 프로젝트는 성공 사례가 실패 사례보다 적은 불균형 데이터이므로, Accuracy value보다 ROC-AUC 등 평가지표를 활용한다.

In [ ]:
# 타깃 분포 확인

target_count = train[TARGET].value_counts().sort_index()
target_ratio = train[TARGET].value_counts(normalize=True).sort_index()

target_summary = pd.DataFrame({
    "count": target_count,
    "ratio": target_ratio,
    "ratio_percent": target_ratio * 100,
})

display(target_summary)

plt.figure(figsize=(7, 4))
ax = sns.barplot(x=target_count.index.astype(str), y=target_count.values, color=main_color)
ax.set_title("임신 성공 여부 분포")
ax.set_xlabel("임신 성공 여부")
ax.set_ylabel("건수")

for i, value in enumerate(target_count.values):
    pct = target_ratio.iloc[i] * 100
    ax.text(i, value, f"{value:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


#### 01-2. 결과 해석

- 74:26 비율로 성공(1)과 실패(0)의 비율 차이가 존재한다.
- 이 데이터는 클래스 불균형 문제를 포함하므로 Accuracy만으로 성능을 판단하지 않는다.
- 이후 모델 평가는 ROC-AUC를 핵심 지표로 사용하고, LogLoss, Recall, F1-score 등을 보조 지표로 확인한다.

* 참고: 성공과 실패를 얼마나 잘 구분하는지 평가하는 `ROC-AUC`를 핵심 지표로 사용한다. 또한 추론의 품질을 확인하기 위해 `LogLoss`, 재현율을 보기 위해 `Recall`과 `F1-score`도 함께 참고한다.

### 01-3. 결측치 현황 확인

전체 변수의 결측치를 확인한다. 결측률이 높은 변수를 무조건 제거하기보다, 결측이 발생한 이유가 시술 프로세스와 관련되어 있는지 확인한다.

In [ ]:
# 결측치 현황 확인

missing = (
    train.isna().sum()
    .to_frame("missing_count")
    .assign(
        missing_rate=lambda x: x["missing_count"] / len(train),
        missing_rate_percent=lambda x: x["missing_rate"] * 100,
    )
    .sort_values("missing_count", ascending=False)
)

display(missing.head(30))

plt.figure(figsize=(9, 8))
plot_df = missing.head(20).sort_values("missing_rate_percent")
ax = sns.barplot(data=plot_df, x="missing_rate_percent", y=plot_df.index, color=main_color)
ax.set_title("결측률 상위 20개 변수") # 전체를 보여주면 너무 많아서 상위 20개만 시각화(EDA)
ax.set_xlabel("결측률(%)")
ax.set_ylabel("")

for i, value in enumerate(plot_df["missing_rate_percent"]):
    ax.text(value + 0.5, i, f"{value:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()


#### 01-3. 결과 해석

전체 결측치 현황을 확인한 결과, 일부 변수에서 매우 높은 결측률이 나타났다.

`난자 해동 경과일`, `PGS 시술 여부`, `PGD 시술 여부`, `착상 전 유전 검사 사용 여부`, `임신 시도 또는 마지막 임신 경과 연수`는 결측률이 약 96~99% 이상으로 매우 높다.  
이러한 변수들은 단순히 값이 누락되었다기보다, 특정 시술 또는 검사 자체가 대부분의 환자에게 수행되지 않았을 가능성이 있다.

또한 `배아 해동 경과일` 역시 결측률이 약 84.3%로 높게 나타났다.  
이는 모든 환자가 동결 배아 해동 과정을 거치는 것이 아니기 때문에 발생한 구조적 결측일 수 있다.

특히 여러 배아·난자 관련 변수에서 결측치 수가 동일하게 6,291건(2.5%)이다.
이는 특정 환자군 또는 특정 시술 유형에서 배아 생성, 배아 이식, 난자 채취 등의 과정이 존재하지 않았기 때문에 발생한 결측일 가능성이 있다.
따라서 결측률이 높다는 이유만으로 변수를 바로 삭제하기보다는, 결측이 발생한 원인을 시술 프로세스 관점에서 해석할 필요가 있다.

이후 전처리 단계에서는 다음 전략을 고려한다.

- 결측 자체를 의미 있는 정보로 보고 missing flag 변수를 생성한다. *핵심*
- 시술 유형별 결측 패턴을 확인하여 구조적 결측 여부를 판단한다.
- 트리 기반 모델에서는 일부 NaN을 그대로 유지하는 전략도 검토한다.
- 선형 모델이나 MLP 계열 모델에서는 결측 대체와 missing flag를 함께 사용한다.

다음 단계에서는 `시술 유형`별 결측 패턴을 확인하여, 결측값이 특정 시술 프로세스와 연결되어 있는지 살펴본다.

### 01-4. 시술 유형별 구조적 결측 확인

DI와 IVF는 시술 프로세스가 다르기 때문에 결측 패턴도 다를 수 있다. 특히 배아 관련 변수의 결측은 단순 누락이 아니라 해당 시술에서 해당 과정이 존재하지 않는다는 의미일 수 있다.


In [ ]:
# 시술 유형별 결측 패턴 확인

treatment_col = "시술 유형"

if treatment_col in train.columns:
    treatment_counts = train[treatment_col].value_counts(dropna=False)
    display(treatment_counts.to_frame("count"))

    missing_by_treatment = (
        train.groupby(treatment_col, dropna=False)
        .apply(lambda x: x.isna().mean())
        .T
    )

    missing_by_treatment["max_missing_rate"] = missing_by_treatment.max(axis=1)
    missing_by_treatment = missing_by_treatment.sort_values("max_missing_rate", ascending=False)

    display(missing_by_treatment.head(30))
else:
    print(f"{treatment_col} 컬럼이 없습니다.")


#### 01-4. 결과 해석

시술 유형별 결측 패턴을 확인한 결과, DI와 IVF 사이에 매우 뚜렷한 차이가 나타났다.

DI 시술에서는 `배아 생성 주요 이유`, `단일 배아 이식 여부`, `총 생성 배아 수`, `이식된 배아 수`, `미세주입된 난자 수`, `해동된 배아 수`, `신선/동결 배아 사용 여부` 등 배아·난자 관련 변수들이 100% 결측으로 나타났다.

반면 IVF 시술에서는 동일한 변수들의 결측률이 대부분 0%에 가깝다.  
이는 DI 시술에서 해당 정보가 단순히 누락된 것이 아니라, 시술 프로세스상 배아 생성·배아 이식·난자 채취 과정이 존재하지 않기 때문에 발생한 구조적 결측으로 해석할 수 있다.

즉, 결측값 자체가 “해당 시술 과정이 수행되지 않았다”는 중요한 정보를 담고 있다.  
따라서 이러한 변수들은 단순 삭제하거나 0으로 일괄 대체하기보다, 결측 여부를 feature로 활용하는 전략이 필요하다.

특히 다음과 같은 전처리 방향을 고려할 수 있다.

- DI 여부를 별도 feature로 명시한다.
- 배아·난자 관련 변수의 결측 여부를 missing flag로 생성한다.
- 트리 기반 모델에서는 구조적 NaN을 그대로 유지하거나, 결측 flag와 함께 사용한다.
- 선형 모델 또는 MLP 계열 모델에서는 중앙값/특정값 대체와 missing flag를 함께 사용한다.
- IVF와 DI의 시술 프로세스가 다르므로, 유형별 모델링 또는 시술 유형별 파생변수를 고려한다.

### 01-5. 주요 변수별 임신 성공률 확인

앞서 확인한 결측 패턴을 바탕으로, 주요 범주형 변수별 임신 성공률을 확인한다.

이 단계에서는 단순히 성공률이 높은 변수를 찾는 것이 아니라, 이후 파생변수 생성과 모델링에 활용할 수 있는 후보 변수를 선별하는 것을 목표로 한다.

In [ ]:
# 주요 범주형 변수별 성공률 확인 함수
def success_rate_by_col(df, col, min_count=0):
    result = (
        df.groupby(col, dropna=False)[TARGET]
        .agg(["count", "mean"])
        .rename(columns={"mean": "success_rate"})
        .assign(success_rate_percent=lambda x: x["success_rate"] * 100)
        .sort_values("success_rate_percent", ascending=False)
    )

    if min_count > 0:
        result = result[result["count"] >= min_count]

    return result


key_cat_cols = [
    "시술 유형",
    "시술 당시 나이",
    "특정 시술 유형",
    "배란 자극 여부",
    "배란 유도 유형",
    "배아 생성 주요 이유",
    "난자 출처",
    "정자 출처",
    "동결 배아 사용 여부",
    "신선 배아 사용 여부",
]

for col in key_cat_cols:
    if col in train.columns:
        print("\n" + "=" * 80)
        print(col)
        display(success_rate_by_col(train, col).head(20))

#### 01-5. 결과 해석

주요 범주형 변수별 임신 성공률을 확인한 결과, 여러 변수에서 target과 뚜렷한 관계가 관찰되었다.

먼저 `시술 유형`에서는 IVF의 임신 성공률이 약 26.2%, DI의 임신 성공률이 약 12.9%로 나타났다.  
이는 IVF와 DI가 서로 다른 시술 프로세스를 가지며, 성공률과 결측 패턴 모두에서 차이가 크다는 것을 의미한다. 따라서 이후 모델링에서는 `시술 유형`을 주요 변수로 활용하고, DI에서 발생하는 구조적 결측을 별도로 고려할 필요가 있다.

`시술 당시 나이`에서는 나이가 증가할수록 전반적으로 임신 성공률이 낮아지는 경향이 확인되었다(관련 논문 추가 #1, #2. 난임과 나이의 인과관계).  
특히 만18-34세의 성공률은 약 32.3%인 반면, 만38-39세는 약 21.7%, 만40-42세는 약 15.9%, 만43-44세는 약 11.8%로 감소한다.  
이는 나이가 임신 성공 여부에 중요한 영향을 미치는 변수임을 보여주며, 이후 나이 구간화 또는 순서형 인코딩을 적용할 수 있다.

`특정 시술 유형`에서는 BLASTOCYST가 포함된 시술 유형의 성공률이 상대적으로 높게 나타났다(시술 유형에 따른 인사이트 #3).  
예를 들어 `IVF / BLASTOCYST`는 약 36.6%, `ICSI / BLASTOCYST`는 약 35.7%의 성공률을 보인다.  
반면 IUI는 약 12.9%로 낮게 나타나며, 이는 DI 계열 시술의 낮은 성공률과도 연결된다. 
따라서 `BLASTOCYST 포함 여부`, `ICSI 포함 여부`, `IVF 포함 여부`, `IUI 여부`와 같은 토큰 기반 파생변수를 생성할 수 있다.

`배아 생성 주요 이유`에서는 `기증용, 현재 시술용`의 성공률이 약 38.0%로 가장 높게 나타났다.  
반면 `배아 저장용`, `기증용`, `난자 저장용` 등 현재 시술 목적이 아닌 항목은 성공률이 거의 0%에 가깝다.  
따라서 `현재 시술용 여부`, `저장용 여부`, `기증용 여부`와 같은 파생변수 생성이 유효할 수 있다.

`난자 출처`에서는 기증 제공의 성공률이 약 31.5%, 본인 제공의 성공률이 약 25.8%로 나타났다.  
이는 기증 난자를 사용하는 경우 환자 본인의 나이보다 난자 자체의 생물학적 조건이 더 중요할 수 있음을 시사한다.  
따라서 이후 `나이 × 난자 출처` 조합 변수 또는 기증 난자 여부 변수를 고려할 필요가 있다.

`정자 출처`는 난자 출처에 비해 성공률 차이가 크지 않지만, 미할당 또는 배우자 및 기증 제공과 같이 표본 수가 작은 범주는 해석에 주의가 필요하다.

`동결 배아 사용 여부`와 `신선 배아 사용 여부`에서는 신선 배아 사용 또는 동결 배아 미사용 그룹의 성공률이 상대적으로 높게 나타났다.  
다만 이 변수들은 시술 프로세스, 배아 이식 여부, 배아 생성 주요 이유와 함께 해석해야 하므로 단독 효과만으로 결론을 내리기보다는 조합 변수로 활용하는 것이 적절하다.
종합하면, 이후 Feature Engineering에서 우선적으로 고려할 변수는 다음과 같다.

- `시술 유형`
- `시술 당시 나이`
- `특정 시술 유형`
- `배아 생성 주요 이유`
- `난자 출처`
- `동결 배아 사용 여부`
- `신선 배아 사용 여부`
- `나이 × 난자 출처`
- `시술 유형 × 특정 시술 유형` *7개 변수, 2개 co-변수 추가 (69+9)*

다음 단계에서는 나이, 난자 수, 배아 수처럼 임상적 의미가 있는 수치형/순서형 변수의 구간화 기준을 탐색한다.

### 01-6. 나이·난자·배아 관련 구간화 기준 탐색

나이, 난자 수, 배아 수는 임신 성공 여부와 직접적인 관련이 있을 가능성이 높은 변수이다.

이 단계에서는 값이 증가할수록 성공률이 단순히 선형적으로 변하는지, 또는 특정 구간에서 급격한 변화나 정체 구간이 나타나는지 확인한다.

이를 바탕으로 이후 전처리 단계에서 임상적 의미를 반영한 구간화 변수를 생성한다.

In [ ]:
# 나이 구간별 성공률 확인: 이거 범주형인데 정렬해서 보면 다를듯
age_order = [
    "만18-34세",
    "만35-37세",
    "만38-39세",
    "만40-42세",
    "만43-44세",
    "만45-50세",
    "알 수 없음"
]

age_success = success_rate_by_col(train, "시술 당시 나이").reset_index()
age_success["시술 당시 나이"] = pd.Categorical(
    age_success["시술 당시 나이"],
    categories=age_order,
    ordered=True
)
age_success = age_success.sort_values("시술 당시 나이")

display(age_success)

plt.figure(figsize=(9, 4))
ax = sns.barplot(
    data=age_success,
    x="시술 당시 나이",
    y="success_rate_percent",
    color=main_color
)

ax.set_title("시술 당시 나이별 임신 성공률")
ax.set_xlabel("시술 당시 나이")
ax.set_ylabel("임신 성공률(%)")
plt.xticks(rotation=20)

for i, row in age_success.reset_index(drop=True).iterrows():
    ax.text(
        i,
        row["success_rate_percent"] + 0.6,
        f'{row["success_rate_percent"]:.1f}%',
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
# 난자 수 구간별 성공률 확인
def make_count_bin(s):
    return pd.cut(
        s,
        bins=[-1, 0, 5, 10, np.inf],
        labels=["0개", "1-5개", "6-10개", "11개 이상"]
    )

oocyte_cols = [
    "수집된 신선 난자 수",
    "혼합된 난자 수",
    "해동 난자 수"
]

for col in oocyte_cols:
    if col in train.columns:
        temp = train[[col, TARGET]].copy()
        temp[f"{col}_구간"] = make_count_bin(temp[col])

        result = (
            temp.groupby(f"{col}_구간", dropna=False)[TARGET]
            .agg(["count", "mean"])
            .rename(columns={"mean": "success_rate"})
            .assign(success_rate_percent=lambda x: x["success_rate"] * 100)
            .reset_index()
        )

        print("\n" + "=" * 80)
        print(col)
        display(result)

        plt.figure(figsize=(7, 4))
        ax = sns.barplot(
            data=result,
            x=f"{col}_구간",
            y="success_rate_percent",
            color=main_color
        )

        ax.set_title(f"{col} 구간별 임신 성공률")
        ax.set_xlabel(f"{col} 구간")
        ax.set_ylabel("임신 성공률(%)")

        for i, row in result.iterrows():
            ax.text(
                i,
                row["success_rate_percent"] + 0.5,
                f'{row["success_rate_percent"]:.1f}%',
                ha="center",
                va="bottom",
                fontsize=9
            )

        plt.tight_layout()
        plt.show()

In [ ]:
# 배아 수 구간별 성공률 확인
def make_embryo_bin(s):
    return pd.cut(
        s,
        bins=[-0.1, 0, 2, 5, np.inf],
        labels=["0개", "1-2개", "3-5개", "6개 이상"],
        include_lowest=True
    )

embryo_cols = [
    "총 생성 배아 수",
    "이식된 배아 수",
    "저장된 배아 수"
]

for col in embryo_cols:
    if col in train.columns:
        temp = train[[col, TARGET]].copy()

        temp[f"{col}_구간"] = make_embryo_bin(temp[col])

        result = (
            temp.groupby(f"{col}_구간", dropna=False, observed=False)[TARGET]
            .agg(["count", "mean"])
            .rename(columns={"mean": "success_rate"})
            .assign(success_rate_percent=lambda x: x["success_rate"] * 100)
            .reset_index()
        )

        # 표에는 결측 그룹도 확인
        display_result = result.copy()

        # 그래프에는 원값이 결측이 아닌 행만 사용
        plot_temp = temp[temp[col].notna()].copy()

        plot_result = (
            plot_temp.groupby(f"{col}_구간", observed=False)[TARGET]
            .agg(["count", "mean"])
            .rename(columns={"mean": "success_rate"})
            .assign(success_rate_percent=lambda x: x["success_rate"] * 100)
            .reset_index()
        )

        print("\n" + "=" * 80)
        print(col)
        display(display_result)

        plt.figure(figsize=(7, 4))
        ax = sns.barplot(
            data=plot_result,
            x=f"{col}_구간",
            y="success_rate_percent",
            color=main_color
        )

        ax.set_title(f"{col} 구간별 임신 성공률")
        ax.set_xlabel(f"{col} 구간")
        ax.set_ylabel("임신 성공률(%)")

        for i, row in plot_result.reset_index(drop=True).iterrows():
            ax.text(
                i,
                row["success_rate_percent"] + 0.5,
                f'{row["success_rate_percent"]:.1f}%',
                ha="center",
                va="bottom",
                fontsize=9
            )

        plt.tight_layout()
        plt.show() # NaN 값이 자꾸 6이상에 달라붙는데 해결 필요

### 01-6. 결과 해석

나이, 난자 수, 배아 수 관련 변수의 구간별 임신 성공률을 확인한 결과, 여러 변수에서 임상적으로 해석 가능한 구간 패턴이 나타났다.

먼저 `시술 당시 나이`에서는 나이가 증가할수록 전반적으로 임신 성공률이 감소하는 경향이 확인되었다. (#1, #2 참고문헌 확인)
만18-34세의 성공률은 약 32.3%, 만35-37세는 약 27.8%, 만38-39세는 약 21.7%로 하락한다.  
특히 만35-37세에서 만38-39세 구간으로 넘어갈 때 성공률이 크게 낮아져, 38세 전후를 중요한 구간 기준으로 고려할 수 있다.  
다만 만45-50세의 성공률이 만43-44세보다 높게 나타나므로, 이 구간은 난자 출처나 기증 난자 사용 여부와 함께 추가 확인이 필요하다.

난자 수 관련 변수에서는 단순히 0개에서 1-5개로 증가한다고 성공률이 바로 높아지는 패턴은 아니었다.  
`수집된 신선 난자 수`와 `혼합된 난자 수` 모두 6-10개 구간부터 성공률이 뚜렷하게 상승하고, 11개 이상 구간에서 가장 높은 성공률을 보였다.  
이는 난자 수가 일정 수준 이상 확보될 때 성공 가능성이 높아지는 경향을 시사한다.  
반면 `해동 난자 수`는 0개에 수렴하고, 1개 이상 구간의 표본 수가 상대적으로 작기 때문에 해석에 주의가 필요하다.

배아 수 관련 변수에서는 `총 생성 배아 수`가 3-5개 이상부터 성공률이 뚜렷하게 상승하고, 6개 이상 구간에서 가장 높은 성공률을 보였다.  
이는 생성된 배아 수가 일정 수준 이상 확보될수록 임신 성공 가능성이 높아지는 경향을 보여준다.

`이식된 배아 수`에서는 0개 구간의 성공률이 거의 0%에 가깝게 나타났다.  
이는 배아 이식이 이루어지지 않은 경우 임신 성공 가능성이 매우 낮다는 점을 보여주는 강한 신호이다.  
반면 1-2개 이식 구간은 약 31.2%의 성공률을 보이며, 3-5개 구간보다 높게 나타났다.  
따라서 이식된 배아 수는 단순히 많을수록 좋은 변수가 아니라, 적정 이식 수가 중요할 수 있다.

`저장된 배아 수`는 1-2개, 3-5개 구간에서 성공률이 약 35~36%로 높게 나타났고, 6개 이상에서는 오히려 낮아졌다.  
이는 저장 배아 수가 일정 수준 이상 많다고 해서 성공률이 계속 증가하지는 않으며, 상단 평탄화 또는 다른 시술 조건과의 조합 효과가 존재할 수 있음을 시사한다.
종합하면, 이후 Feature Engineering에서는 다음 구간화 변수를 우선 고려한다. (#3 참고자료-용어사전- 확인)

- 나이 구간: 38세 전후, 40세 이상, 43세 이상 여부
- 난자 수 구간: 0개, 1-5개, 6-10개, 11개 이상
- 총 생성 배아 수 구간: 0개, 1-2개, 3-5개, 6개 이상
- 이식된 배아 수 구간: 0개, 1-2개, 3개 이상
- 저장된 배아 수 구간: 0개, 1-2개, 3-5개, 6개 이상

또한 각 변수의 NaN 그룹은 대부분 DI 시술과 연결된 구조적 결측일 가능성이 있으므로, 구간화와 별도로 missing flag 변수로 활용하는 것이 적절하다.
다음 단계에서는 EDA 결과를 바탕으로, 임상 지식 기반 결측 유추와 구조적 결측 flag 생성 방향을 정리한다. *+5개*

#### 제언: 임상에서의 결측치 유추 파생변수 확인

앞선 EDA에서 일부 변수의 결측이 단순 누락이 아니라 시술 프로세스와 연결된 구조적 결측일 가능성을 확인하였다.

이번 단계에서는 임상적 의미를 바탕으로 결측값을 유추할 수 있는 후보 규칙을 탐색한다.

예를 들어 해동된 배아 수가 1개 이상이면 동결 배아 이식(FER) 과정이 있었을 가능성이 있고, 미세주입된 난자 수가 1개 이상이면 ICSI 관련 시술이 수행되었을 가능성이 있다. *3기 1위 자료확인*

이러한 규칙은 바로 확정적으로 적용하기보다, train 데이터에서 해당 조건의 표본 수와 성공률을 먼저 확인한 뒤 전처리 단계에서 적용 여부를 판단한다.

In [ ]:
# 결측치 유추 파생변수 확인
def rule_summary(df, mask, rule_name):
    subset = df.loc[mask]

    return {
        "rule": rule_name,
        "count": len(subset),
        "success_rate": subset[TARGET].mean(),
        "success_rate_percent": subset[TARGET].mean() * 100
    }


proc = train["특정 시술 유형"].astype("string")
ovulation_type = train["배란 유도 유형"].astype("string")

unknown_proc = proc.isna() | proc.isin(["Unknown", "알 수 없음"])
unknown_ovulation_type = ovulation_type.isna() | ovulation_type.isin(["Unknown", "알 수 없음"])

thawed_embryo_ge1 = pd.to_numeric(train["해동된 배아 수"], errors="coerce").fillna(0) >= 1
micro_oocyte_ge1 = pd.to_numeric(train["미세주입된 난자 수"], errors="coerce").fillna(0) >= 1
ovulation_stim_zero = pd.to_numeric(train["배란 자극 여부"], errors="coerce").fillna(-1) == 0

candidate_rules = [
    rule_summary(
        train,
        thawed_embryo_ge1,
        "해동된 배아 수 1개 이상: FER 가능성"
    ),
    rule_summary(
        train,
        thawed_embryo_ge1 & unknown_proc,
        "특정 시술 유형 Unknown + 해동된 배아 수 1개 이상: FER 유추 후보"
    ),
    rule_summary(
        train,
        micro_oocyte_ge1,
        "미세주입된 난자 수 1개 이상: ICSI 가능성"
    ),
    rule_summary(
        train,
        micro_oocyte_ge1 & unknown_proc,
        "특정 시술 유형 Unknown + 미세주입된 난자 수 1개 이상: ICSI 유추 후보"
    ),
    rule_summary(
        train,
        ovulation_stim_zero & unknown_ovulation_type,
        "배란 자극 여부 0 + 배란 유도 유형 Unknown: 자연주기 유추 후보"
    ),
]

candidate_rule_df = pd.DataFrame(candidate_rules)
display(candidate_rule_df)

In [ ]:
# 유추 후보 조건별 기존 시술 유형 분포 확인
print("해동된 배아 수 1개 이상인 경우의 특정 시술 유형 분포")
display(
    train.loc[thawed_embryo_ge1, "특정 시술 유형"]
    .value_counts(dropna=False)
    .head(20)
    .to_frame("count")
)

print("미세주입된 난자 수 1개 이상인 경우의 특정 시술 유형 분포")
display(
    train.loc[micro_oocyte_ge1, "특정 시술 유형"]
    .value_counts(dropna=False)
    .head(20)
    .to_frame("count")
)

print("배란 자극 여부 0인 경우의 배란 유도 유형 분포")
display(
    train.loc[ovulation_stim_zero, "배란 유도 유형"]
    .value_counts(dropna=False)
    .head(20)
    .to_frame("count")
)

#### 제언에 대한 재탐색 결과

임상 지식 기반 결측 유추 후보를 확인한 결과, 일부 조건은 충분한 표본 수를 가지며 시술 프로세스와 연결될 가능성이 확인되었다.

먼저 `해동된 배아 수`가 1개 이상인 경우는 40,385건이며, 성공률은 약 22.9%로 나타났다.  
해동된 배아가 존재한다는 것은 과거에 생성 및 동결 보존된 배아를 해동한 과정이 있었다는 의미이므로, 동결 배아 이식(FER) 관련 시술일 가능성이 있다.

특히 `특정 시술 유형`이 Unknown이면서 `해동된 배아 수`가 1개 이상인 경우가 26,924건으로 비교적 많게 나타났다.  
이는 Unknown으로 기록된 특정 시술 유형 중 상당수가 실제로는 FER 관련 시술일 가능성을 시사한다.  
따라서 전처리 단계에서 이 조건을 활용해 `특정 시술 유형`을 보완하거나, 별도의 `FER 유추 flag`를 생성하는 전략을 검토할 수 있다.

`미세주입된 난자 수`가 1개 이상인 경우는 122,391건이며, 성공률은 약 27.3%로 나타났다.  
이는 미세주입 과정이 수행된 경우로 해석할 수 있으며, ICSI 관련 시술과 연결될 가능성이 높다.

다만 `특정 시술 유형`이 Unknown이면서 `미세주입된 난자 수`가 1개 이상인 경우는 74건으로 표본 수가 매우 작다.  
따라서 이 조건을 이용해 Unknown을 ICSI로 직접 대체하는 전략은 효과가 제한적일 수 있으며, 대체보다는 flag 변수로만 활용하는 것이 더 안전하다.

`배란 자극 여부`가 0이면서 `배란 유도 유형`이 Unknown인 경우는 58,631건으로 많고, 성공률은 약 23.2%로 나타났다.  
배란 자극이 없다는 것은 인위적인 호르몬 자극 없이 자연 주기 또는 비자극 방식으로 진행되었을 가능성을 의미한다.  
따라서 이 조건은 `자연주기 유추 flag` 또는 `배란 유도 유형 보완` 후보로 검토할 수 있다.

종합하면, 전처리 단계에서 우선 검토할 임상 지식 기반 변수는 다음과 같다.

- `해동된 배아 수 >= 1` 기반 FER 유추 flag
- `특정 시술 유형 Unknown` + `해동된 배아 수 >= 1` 기반 특정 시술 유형 보완 후보
- `미세주입된 난자 수 >= 1` 기반 ICSI 관련 flag
- `배란 자극 여부 == 0` + `배란 유도 유형 Unknown` 기반 자연주기 유추 flag

다만 이러한 규칙은 도메인 기반 유추이므로, 원본 값을 무조건 대체하기보다는 먼저 flag 변수로 생성하고 validation 또는 OOF ROC-AUC 개선 여부를 확인하는 방식이 안전하다.

다음 단계에서는 지금까지의 EDA 결과를 종합하여 전처리 및 Feature Engineering 방향을 정리한다. *+4*

### 01-7. 이상치 후보 확인

수치형 변수에서 일반적인 범위를 벗어나는 값이 있는지 확인한다.

의료 및 시술 데이터에서는 극단값이 실제 기록일 수 있으므로, 이상치를 바로 제거하지 않고 먼저 분포와 발생 비율을 확인한다.  
이후 전처리 단계에서 clipping, 구간화, flag 생성 여부를 판단한다.

In [ ]:
# 수치형 변수의 기초 통계 확인
numeric_cols_raw = train.select_dtypes(include=[np.number]).columns.drop(TARGET, errors="ignore")

num_summary = train[numeric_cols_raw].describe().T
num_summary["missing_count"] = train[numeric_cols_raw].isna().sum()
num_summary["missing_rate"] = train[numeric_cols_raw].isna().mean()

display(num_summary.sort_values("missing_rate", ascending=False).head(30))

In [ ]:
# IQR 기준 이상치 후보 계산 및 시각화
numeric_cols_raw = train.select_dtypes(include=[np.number]).columns.drop(TARGET, errors="ignore")

outlier_rows = []

for col in numeric_cols_raw:
    s = train[col].dropna()

    if len(s) == 0:
        continue

    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1

    if iqr == 0:
        continue

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_mask = (train[col] < lower) | (train[col] > upper)

    outlier_rows.append({
        "column": col,
        "outlier_count": int(outlier_mask.sum()),
        "outlier_rate": outlier_mask.mean(),
        "outlier_rate_percent": outlier_mask.mean() * 100,
        "lower_bound": lower,
        "upper_bound": upper,
        "min": s.min(),
        "max": s.max(),
    })

outlier_df = (
    pd.DataFrame(outlier_rows)
    .sort_values("outlier_rate_percent", ascending=False)
)

display(outlier_df.head(30))

plot_outlier = outlier_df.head(15).sort_values("outlier_rate_percent")

plt.figure(figsize=(9, 6))
ax = sns.barplot(
    data=plot_outlier,
    x="outlier_rate_percent",
    y="column",
    color=main_color
)

ax.set_title("IQR 기준 이상치 후보 비율 상위 변수")
ax.set_xlabel("이상치 후보 비율(%)")
ax.set_ylabel("")

for i, value in enumerate(plot_outlier["outlier_rate_percent"]):
    ax.text(value + 0.2, i, f"{value:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()

#### 01-7. 결과 해석

수치형 변수에 대해 IQR 기준 이상치 후보를 확인한 결과, 이상치 후보는 주로 배아·난자 수량 관련 변수에서 나타났다.

가장 높은 이상치 후보 비율을 보인 변수는 `저장된 배아 수`로, 약 5.5%의 행이 IQR 기준 상한값을 초과하였다.  
그 외에도 `미세주입에서 생성된 배아 수`, `미세주입된 난자 수`, `혼합된 난자 수`, `총 생성 배아 수`, `수집된 신선 난자 수` 등 난자 및 배아 수량 관련 변수에서 일부 이상치 후보가 확인되었다.

이 변수들의 최댓값은 대부분 51까지 나타나며, IQR 기준으로는 극단값으로 분류될 수 있다.  
다만 난임 시술 데이터에서 난자 또는 배아 수가 많은 경우는 실제 임상 기록일 수 있으므로, 이를 단순 오류값으로 보고 제거하는 것은 적절하지 않다.

또한 `이식된 배아 수`, `미세주입 배아 이식 수`, `배아 이식 경과일` 등은 IQR 기준 이상치 후보가 거의 없거나 0건으로 나타났다.  
이는 해당 변수들이 비교적 제한된 범위 안에서 기록되어 있음을 의미한다.

따라서 본 프로젝트에서는 이상치 후보를 즉시 제거하지 않고, 다음과 같은 방식으로 활용하는 것이 적절하다.

- 난자·배아 수량 변수는 원본 값을 유지한다.
- 극단값이 존재하는 변수는 구간화 변수를 함께 사용한다.
- 필요 시 상위 구간 여부를 나타내는 flag 변수를 생성한다.
- 트리 기반 모델은 이상치에 비교적 강하므로, 우선 원본 값을 유지하고 모델 성능을 확인한다.
- 이후 Feature Importance와 OOF ROC-AUC를 기준으로 clipping 또는 pruning 여부를 판단한다.

결론적으로, 현재 확인된 이상치 후보는 대부분 실제 시술 과정에서 발생 가능한 수량값으로 판단되며, 일괄 삭제보다는 구간화 및 모델 검증을 통해 처리하는 것이 적절하다.

### 01-8. 수치형 변수 간 상관관계 확인

수치형 변수들이 `임신 성공 여부`와 어떤 방향의 관계를 가지는지 확인한다.  
또한 변수들끼리 상관관계가 지나치게 높은 경우, 이후 모델링 단계에서 중복 정보 또는 noise pruning 후보로 검토할 수 있다.

In [ ]:
# 수치형 변수와 target 간 상관관계 확인
TARGET = "임신 성공 여부"

numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != TARGET]

corr_with_target = (
    train[numeric_cols + [TARGET]]
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET)
    .dropna()
    .sort_values(key=lambda x: x.abs(), ascending=False)
    .reset_index()
)

corr_with_target.columns = ["feature", "corr_with_target"]
corr_with_target["abs_corr"] = corr_with_target["corr_with_target"].abs()

display(corr_with_target.head(30))

In [ ]:
# target과 상관관계가 높은 변수 중심 heatmap
top_corr_features = corr_with_target.head(20)["feature"].tolist()
corr_matrix = train[top_corr_features + [TARGET]].corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    cmap="vlag",
    center=0,
    linewidths=0.3,
    linecolor="#333333"
)

plt.title("Target 관련 주요 수치형 변수 상관관계")
plt.tight_layout()
plt.show()

In [ ]:
# 변수 간 상관관계가 높은 조합 확인
feature_corr = train[top_corr_features].corr(numeric_only=True).abs()

high_corr_pairs = []

for i, col1 in enumerate(feature_corr.columns):
    for col2 in feature_corr.columns[i + 1:]:
        corr_value = feature_corr.loc[col1, col2]
        if corr_value >= 0.8:
            high_corr_pairs.append({
                "feature_1": col1,
                "feature_2": col2,
                "abs_corr": corr_value
            })

high_corr_df = pd.DataFrame(high_corr_pairs)

if len(high_corr_df) > 0:
    high_corr_df = high_corr_df.sort_values("abs_corr", ascending=False)

display(high_corr_df.head(30))

#### 01-8. 결과 해석

- `임신 성공 여부`와 가장 상관관계가 높은 수치형 변수는 `이식된 배아 수`, `배아 이식 경과일`, `총 생성 배아 수`, `단일 배아 이식 여부`, `혼합된 난자 수`였다.
- 상위 변수들이 대부분 난자·배아 생성·이식 과정과 관련되어 있어, 임신 성공 여부는 단순 인구통계 변수보다 실제 시술 진행 결과와 더 직접적으로 연결되어 있음을 확인할 수 있다.
- 다만 가장 높은 상관계수도 약 0.16 수준이므로, 단일 변수만으로는 임신 성공 여부를 강하게 설명하기 어렵다. 따라서 모델링 단계에서는 변수 간 조합, 구간화, 결측 flag, 비율 변수 등을 함께 활용할 필요가 있다.
- `임신 시도 또는 마지막 임신 경과 연수`는 음의 상관관계를 보였으며, 기간이 길수록 임신 성공률이 낮아지는 방향성을 보인다.
- `동결 배아 사용 여부`와 `신선 배아 사용 여부`는 상관계수가 약 0.997로 매우 높아 사실상 중복 정보에 가깝다. 이후 Feature Importance 또는 OOF 성능 비교를 통해 하나만 남기거나 조합 변수로 재구성할 수 있다.
- `미세주입된 난자 수`와 `미세주입에서 생성된 배아 수`, `혼합된 난자 수`와 `파트너 정자와 혼합된 난자 수`도 높은 상관관계를 보여 중복 또는 파생 비율 변수 후보로 판단된다.
- 초기 모델에서는 변수 제거를 바로 수행하지 않고, 이후 모델링 결과와 Feature Importance를 기준으로 noise pruning 여부를 결정한다.

### 01-9. EDA 종합 정리 및 전처리 방향 제시

EDA를 통해 확인한 핵심 내용과 이후 전처리 방향은 다음과 같다.

1. 데이터 구조
- train 데이터에는 target인 `임신 성공 여부`가 포함되어 있고, test 데이터에는 target이 없다.
- train과 test의 컬럼 차이는 target 컬럼 하나뿐이며, 기본 데이터 구조는 정상적이다.
- 최종 목표는 test 데이터의 `임신 성공 여부` 확률을 예측하는 것이다.

2. 타깃 분포
- 임신 실패(0)는 약 74.2%, 임신 성공(1)은 약 25.8%로 나타났다.
- 성공 클래스가 상대적으로 적은 불균형 이진 분류 문제이다.
- 따라서 Accuracy보다 ROC-AUC를 핵심 평가 지표로 사용하고, LogLoss, Recall, F1-score를 보조 지표로 함께 확인한다.

3. 결측 패턴
- 일부 변수는 결측률이 90% 이상으로 매우 높게 나타났다.
- DI 시술에서는 배아·난자·이식 관련 변수의 결측률이 100%에 가깝게 나타났다.
- 이는 단순 누락이 아니라 시술 프로세스 차이에 따른 구조적 결측으로 해석할 수 있다.
- 따라서 결측값은 일괄 삭제하거나 평균값으로 대체하기보다, 결측 여부 자체를 feature로 활용할 필요가 있다.

4. 주요 범주형 변수
- `시술 유형`에서는 IVF가 DI보다 성공률이 높게 나타났다.
- `시술 당시 나이`는 성공률과 뚜렷한 관련이 있으며, 나이가 증가할수록 전반적으로 성공률이 낮아졌다.
- `특정 시술 유형`에서는 BLASTOCYST가 포함된 유형의 성공률이 상대적으로 높았다.
- `배아 생성 주요 이유`, `난자 출처`, `정자 출처`, `신선/동결 배아 사용 여부`도 성공률 차이를 보였다.
- 범주형 변수 간 조합이 의미를 가질 가능성이 있어, 이후 조합 변수를 생성한다.

5. 나이·난자·배아 구간 패턴
- 나이에서는 38세 전후로 성공률 하락 폭이 커지는 패턴이 확인되었다.
- 40세 이상, 43세 이상 구간에서도 추가적인 성공률 하락이 나타났다.
- 난자 수는 6개 이상 또는 11개 이상 구간에서 성공률이 높아지는 경향을 보였다.
- 총 생성 배아 수는 3개 이상부터 성공률이 상승하는 경향을 보였다.
- 이식된 배아 수는 단순히 많을수록 좋은 것이 아니라, 1-2개 구간에서 성공률이 높게 나타났다.
- 저장된 배아 수는 일정 수준 이후 성공률이 더 이상 증가하지 않는 패턴을 보였다.
- 따라서 수량형 변수는 원본값뿐 아니라 구간화 변수를 함께 활용한다.

6. 임상 지식 기반 결측 유추
- `해동된 배아 수 >= 1`인 경우 FER 관련 시술 가능성이 있다.
- `미세주입된 난자 수 >= 1`인 경우 ICSI 관련 시술 가능성이 있다.
- `배란 자극 여부 == 0`인 경우 자연주기 또는 비자극 방식 가능성이 있다.
- 단, 이러한 유추는 원본 값을 직접 대체하기보다 flag 변수로 먼저 활용하는 것이 안전하다.

7. 이상치 후보
- IQR 기준 이상치 후보는 주로 난자 수, 배아 수, 혼합 난자 수 등 수량형 변수에서 나타났다.
- 최대값 51처럼 큰 값이 존재하지만, 의료 시술 과정에서 발생 가능한 실제 기록일 수 있다.
- 따라서 이상치 후보를 일괄 삭제하지 않고, 구간화 또는 clipping 여부를 모델 성능 기준으로 판단한다.

8. 수치형 변수 간 상관관계
- `이식된 배아 수`, `배아 이식 경과일`, `총 생성 배아 수`, `단일 배아 이식 여부`, `혼합된 난자 수`가 target과 상대적으로 높은 상관관계를 보였다.
- 다만 가장 높은 상관계수도 약 0.16 수준으로, 단일 변수만으로는 임신 성공 여부를 강하게 설명하기 어렵다.
- `동결 배아 사용 여부`와 `신선 배아 사용 여부`처럼 서로 매우 높은 상관관계를 가진 변수들이 존재한다.
- 이러한 변수들은 이후 Feature Importance와 OOF 성능을 기준으로 noise pruning 여부를 검토한다.

9. 전처리 및 Feature Engineering 방향
- 전처리 기준은 train 데이터에서 산출하고, test 데이터에는 동일한 규칙을 적용한다.
- target과 ID는 별도로 보존한다.
- 구조적 결측 가능성이 있는 변수는 train에서 확인한 기준과 도메인 규칙에 따라 missing flag를 생성한다.
- 나이는 순서형 변수로 변환하고, 38세 이상·40세 이상·43세 이상 flag를 생성한다.
- 난자 수, 배아 수, 이식 배아 수는 성공률 패턴을 반영하여 구간화한다.
- FER, ICSI, 자연주기 가능성을 나타내는 임상 지식 기반 flag를 생성한다.
- `나이 x 난자 출처`, `시술 유형 x 특정 시술 유형`, `배아 생성 주요 이유 x 특정 시술 유형` 등 조합 변수를 생성한다.
- 초기에는 후보 변수를 넓게 사용하고, 모델링 이후 중요도가 낮은 변수는 성능 검증을 통해 제거한다.

10. 모델링 방향
- 범주형 변수가 많고 결측 자체가 의미를 가질 수 있으므로 CatBoost를 주요 후보 모델로 사용한다.
- LightGBM, XGBoost도 함께 비교하여 트리 기반 모델 간 성능 차이를 확인한다.
- K-Fold OOF 검증을 통해 모델의 일반화 성능을 안정적으로 평가한다.
- 최종 단계에서는 단일 모델보다 여러 모델의 예측값을 조합하는 ensemble을 검토한다.


## 02. 데이터 전처리 및 Feature Engineering

EDA 결과를 바탕으로 모델 학습에 사용할 feature를 구성한다.  
결측치와 이상치는 단순 삭제하지 않고, 시술 과정의 차이와 변수별 의미를 고려하여 처리한다.

또한 나이, 난자 수, 배아 수처럼 성공률 차이가 뚜렷했던 변수는 구간화하고, 시술 과정과 관련된 조합 변수 및 비율 변수를 생성한다.

진행 순서는 다음과 같다.

1. 학습용 데이터 구성
2. 결측치 처리: 구조적 결측 및 일반 결측 flag 생성
3. 이상치 처리: clipping 후보 생성
4. 임상 지식 기반 파생변수 생성
5. 나이·난자·배아 구간화 변수 생성
6. 조합 변수 및 비율 변수 생성
7. 범주형/수치형 변수 구분
8. 인코딩 전략 정리
9. 스케일링 전략 정리
10. 학습/검증 데이터 분리

### 02-1. 학습용 데이터 구성

모델 학습에 사용할 feature와 target을 분리한다.  
target인 `임신 성공 여부`는 예측해야 할 값이므로 `y`로 따로 저장하고, 나머지 변수들을 학습용 feature로 사용한다.


In [ ]:
# 학습용 feature와 target 분리
train_raw = train.copy()
test_raw = test.copy()

y = train_raw[TARGET].copy()

X = train_raw.drop(columns=[TARGET]).copy()
X_test = test_raw.copy()

train_id = X[ID_COL].copy()
test_id = X_test[ID_COL].copy()

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

assert TARGET not in X.columns
assert TARGET not in X_test.columns
assert list(X.columns) == list(X_test.columns)

#### 02-1. 처리 방식 설명

- train 데이터에서 target인 `임신 성공 여부`를 분리하여 `y`로 저장하였다.
- target을 제외한 변수들은 학습용 feature인 `X`로 저장하였다.
- ID는 모델 학습에는 사용하지 않고, 제출 파일 생성 시 활용하기 위해 별도로 보존하였다.
- 제출용 데이터는 이후 동일한 feature 생성 규칙을 적용하기 위해 `X_test`로 보존하였다.

### 02-2. 결측치 처리: 구조적 결측 및 일반 결측 flag 생성

EDA에서 DI 시술의 배아·난자·이식 관련 변수들이 구조적으로 결측되는 패턴을 확인하였다.  
또한 IVF에서도 유전검사, 경과일, 임신 경과 연수 등 일부 변수에 일반 결측이 존재한다.

따라서 결측값을 행 삭제 방식으로 처리하지 않고, 결측 여부를 나타내는 flag를 생성한다.

In [ ]:
# 구조적 결측 가능성이 높은 배아·난자·이식 관련 변수
process_missing_cols = [
    "배아 생성 주요 이유",
    "단일 배아 이식 여부",
    "착상 전 유전 진단 사용 여부",
    "총 생성 배아 수",
    "미세주입된 난자 수",
    "미세주입에서 생성된 배아 수",
    "이식된 배아 수",
    "미세주입 배아 이식 수",
    "저장된 배아 수",
    "미세주입 후 저장된 배아 수",
    "해동된 배아 수",
    "해동 난자 수",
    "수집된 신선 난자 수",
    "저장된 신선 난자 수",
    "혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수",
    "기증자 정자와 혼합된 난자 수",
    "동결 배아 사용 여부",
    "신선 배아 사용 여부",
]

process_missing_cols = [col for col in process_missing_cols if col in X.columns]

for df in [X, X_test]:
    for col in process_missing_cols:
        flag_col = col + "_missing_flag"
        df[flag_col] = df[col].isna().astype(int)

    process_flag_cols = [col + "_missing_flag" for col in process_missing_cols]
    df["process_missing_count"] = df[process_flag_cols].sum(axis=1)
    df["process_missing_rate"] = df["process_missing_count"] / len(process_flag_cols)

print("구조적 결측 대상 변수 수:", len(process_missing_cols))
print("생성된 구조적 missing flag 수:", len(process_missing_cols))

display(X[["process_missing_count", "process_missing_rate"]].describe())


In [ ]:
# train 기준 구조적 결측 패턴과 target 관계 확인
train_missing_check = X[["시술 유형", "process_missing_count"]].copy()
train_missing_check[TARGET] = y.values

process_missing_summary = (
    train_missing_check
    .groupby("process_missing_count")[TARGET]
    .agg(count="count", success_rate="mean")
    .reset_index()
)

process_missing_summary["success_rate_percent"] = process_missing_summary["success_rate"] * 100

display(process_missing_summary)

process_missing_by_treatment = (
    train_missing_check
    .groupby("시술 유형")["process_missing_count"]
    .agg(count="count", mean="mean", median="median", min="min", max="max")
)

display(process_missing_by_treatment)


In [ ]:
# 일반 결측 변수에 대한 missing flag 생성
missing_cols = [
    col for col in X.columns
    if col != ID_COL
    and col not in process_missing_cols
    and X[col].isna().sum() > 0
]

general_missing_flag_cols = []

for col in missing_cols:
    flag_col = col + "_is_missing"
    X[flag_col] = X[col].isna().astype(int)
    X_test[flag_col] = X_test[col].isna().astype(int)
    general_missing_flag_cols.append(flag_col)

missing_flag_summary = pd.DataFrame({
    "column": missing_cols,
    "train_missing_count": [X[col].isna().sum() for col in missing_cols],
    "train_missing_rate": [X[col].isna().mean() for col in missing_cols],
})

missing_flag_summary["train_missing_rate_percent"] = missing_flag_summary["train_missing_rate"] * 100
missing_flag_summary = missing_flag_summary.sort_values("train_missing_count", ascending=False)

print("일반 결측 변수 수:", len(missing_cols))
print("생성된 일반 missing flag 수:", len(general_missing_flag_cols))

display(missing_flag_summary.head(30))

#### 02-2. 결과 해석

- 배아·난자·이식 관련 19개 변수를 구조적 결측 후보로 지정하고, 각 변수의 결측 여부를 missing flag로 생성하였다.
- `process_missing_count`는 0 또는 19로만 나타났다.
- `process_missing_count = 0`인 행은 250,060건이며 성공률은 약 26.16%였다.
- `process_missing_count = 19`인 행은 6,291건이며 성공률은 약 12.89%였다.
- 시술 유형별로 확인한 결과, IVF는 구조적 결측 개수가 모두 0개였고 DI는 모두 19개였다.
- 따라서 이 19개 변수의 동시 결측은 단순 누락이 아니라 DI 시술에서 배아 생성·이식 과정이 존재하지 않아 발생한 구조적 결측으로 해석할 수 있다.
- DI의 결측값을 0이나 평균값으로 대체하면 “해당 과정이 존재하지 않음”이라는 의미가 사라질 수 있으므로, 원본 NaN을 유지하고 missing flag를 함께 사용한다.

- 구조적 결측 외에도 train 데이터에서 일반 결측이 존재하는 변수는 12개였다.
- 특히 `난자 해동 경과일`, `PGS 시술 여부`, `PGD 시술 여부`, `착상 전 유전 검사 사용 여부`, `임신 시도 또는 마지막 임신 경과 연수`는 결측률이 96% 이상으로 매우 높았다.
- 이러한 변수들은 모든 환자에게 항상 해당되는 정보가 아니라, 특정 조건에서만 기록되는 정보일 가능성이 있다.
- 따라서 일반 결측 변수도 행 삭제 방식으로 처리하지 않고, 변수별 missing flag를 생성하였다.
- 이후 트리 기반 모델에서는 원본 NaN과 missing flag를 함께 활용한다.

### 02-3. 이상치 처리

EDA에서 난자 수, 배아 수, 혼합 난자 수 등 일부 수량형 변수에서 IQR 기준 이상치 후보가 확인되었다.  
그러나 의료 시술 데이터에서는 큰 값이 실제 기록일 수 있으므로, 이상치 후보를 일괄 삭제하지 않는다.

대신 원본 변수는 유지하고, train 데이터에서 계산한 IQR 기준으로 clipping 후보 변수를 별도로 생성한다.


In [ ]:
# train 기준 IQR로 clipping 후보 변수 생성
outlier_candidate_cols = [
    "저장된 배아 수",
    "미세주입에서 생성된 배아 수",
    "미세주입된 난자 수",
    "혼합된 난자 수",
    "총 생성 배아 수",
    "수집된 신선 난자 수",
    "파트너 정자와 혼합된 난자 수",
    "임신 시도 또는 마지막 임신 경과 연수",
]

outlier_candidate_cols = [col for col in outlier_candidate_cols if col in X.columns]
clipping_summary = []
clip_bounds = {}

for col in outlier_candidate_cols:
    train_values = pd.to_numeric(X[col], errors="coerce")

    q1 = train_values.quantile(0.25)
    q3 = train_values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    clip_bounds[col] = (lower_bound, upper_bound)

    clipped_col = col + "_clipped"
    X[clipped_col] = pd.to_numeric(X[col], errors="coerce").clip(lower_bound, upper_bound)
    X_test[clipped_col] = pd.to_numeric(X_test[col], errors="coerce").clip(lower_bound, upper_bound)

    clipping_summary.append({
        "column": col,
        "clipped_column": clipped_col,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "train_original_min": train_values.min(),
        "train_original_max": train_values.max(),
        "train_clipped_min": X[clipped_col].min(),
        "train_clipped_max": X[clipped_col].max(),
    })

clipping_summary = pd.DataFrame(clipping_summary)
clipped_cols = [col + "_clipped" for col in outlier_candidate_cols]

display(clipping_summary)
print("생성된 clipping 후보 변수 수:", len(clipped_cols))

#### 02-3. 결과 해석

- EDA에서 이상치 후보로 확인된 8개 수량형 변수에 대해 clipping 후보 변수를 생성하였다.
- `저장된 배아 수`, `미세주입된 난자 수`, `혼합된 난자 수`, `총 생성 배아 수`, `수집된 신선 난자 수` 등은 원본 최대값이 51까지 나타났다.
- IQR 기준을 적용하면 각 변수의 상한값은 `저장된 배아 수` 5.0, `미세주입에서 생성된 배아 수` 12.5, `미세주입된 난자 수` 17.5, `혼합된 난자 수` 23.0, `총 생성 배아 수` 18.5 등으로 조정된다.
- `임신 시도 또는 마지막 임신 경과 연수`는 원본 범위가 0~20년이었고, clipping 후보 변수에서는 1~17년 범위로 조정되었다.
- 원본 변수는 삭제하지 않고 유지하였으며, clipping 변수는 모델 성능 비교를 위한 후보 feature로 추가하였다.
- 의료 시술 데이터에서는 큰 값이 실제 기록일 수 있으므로, 이상치 후보를 전처리 단계에서 임의로 제거하지 않는다.
- 이후 모델링 단계에서 원본 변수와 clipping 변수의 Feature Importance 및 OOF 성능을 비교하여 최종 사용 여부를 판단한다.

### 02-4. 임상 지식 기반 파생변수 생성

EDA에서 확인한 시술 프로세스 단서를 바탕으로 FER, ICSI, 자연주기 가능성을 나타내는 flag를 생성한다.


In [ ]:
# 임상 지식 기반 flag 생성 함수
def to_numeric_safe_series(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(np.nan, index=df.index)


def add_clinical_inference_flags(df):
    specific_proc = df["특정 시술 유형"].astype("string").fillna("Unknown")

    df["fer_inferred_flag"] = (
        to_numeric_safe_series(df, "해동된 배아 수").fillna(0) >= 1
    ).astype(int)

    df["icsi_inferred_flag"] = (
        to_numeric_safe_series(df, "미세주입된 난자 수").fillna(0) >= 1
    ).astype(int)

    df["natural_cycle_inferred_flag"] = (
        to_numeric_safe_series(df, "배란 자극 여부").fillna(-1) == 0
    ).astype(int)

    df["unknown_proc_with_thawed_embryo_flag"] = (
        specific_proc.isin(["Unknown", "알 수 없음"])
        & (to_numeric_safe_series(df, "해동된 배아 수").fillna(0) >= 1)
    ).astype(int)

    df["unknown_proc_with_icsi_oocyte_flag"] = (
        specific_proc.isin(["Unknown", "알 수 없음"])
        & (to_numeric_safe_series(df, "미세주입된 난자 수").fillna(0) >= 1)
    ).astype(int)

    return df


X = add_clinical_inference_flags(X)
X_test = add_clinical_inference_flags(X_test)

inferred_cols = [
    "fer_inferred_flag",
    "icsi_inferred_flag",
    "natural_cycle_inferred_flag",
    "unknown_proc_with_thawed_embryo_flag",
    "unknown_proc_with_icsi_oocyte_flag",
]

print("임상 지식 기반 flag 생성 완료")
display(X[inferred_cols].sum().to_frame("train_count"))


In [ ]:
# train-based success rate by clinical inference flag
flag_success_summary = []

for col in inferred_cols:
    temp = pd.DataFrame({
        "feature": col,
        "value": X[col],
        TARGET: y.values,
    })

    summary = (
        temp.groupby(["feature", "value"], dropna=False)[TARGET]
        .agg(count="count", success_rate="mean")
        .reset_index()
    )
    summary["success_rate_percent"] = summary["success_rate"] * 100
    flag_success_summary.append(summary)

flag_success_summary = pd.concat(flag_success_summary, ignore_index=True)
display(flag_success_summary)

#### 02-4. 결과 해석

- 임상 지식 기반으로 FER, ICSI, 자연주기 가능성을 나타내는 flag 변수를 생성하였다.
- `fer_inferred_flag`는 40,385건에서 1로 나타났으며, 해당 그룹의 성공률은 약 22.85%였다.
- `icsi_inferred_flag`는 122,391건에서 1로 나타났으며, 해당 그룹의 성공률은 약 27.29%였다.
- `natural_cycle_inferred_flag`는 58,631건에서 1로 나타났으며, 해당 그룹의 성공률은 약 23.17%였다.
- `특정 시술 유형`이 Unknown이면서 해동된 배아 수가 1개 이상인 경우는 26,924건으로, FER 유추 후보로 볼 수 있다.
- `특정 시술 유형`이 Unknown이면서 미세주입된 난자 수가 1개 이상인 경우는 74건으로 매우 적어, 단독 변수로 해석할 때는 주의가 필요하다.

- 이 단계에서는 원본 값을 직접 수정하지 않고, 임상적으로 해석 가능한 조건을 별도 flag로 추가하였다.
- 예를 들어 `해동된 배아 수 >= 1`이라는 관측값은 해당 행에서 이미 주어진 정보이므로, 이를 바탕으로 FER 가능성을 나타내는 flag를 만드는 것은 새로운 외부 정보를 추가하는 것이 아니다.
- 또한 성공 여부(`임신 성공 여부`)를 사용해 값을 채우거나 변환하지 않았기 때문에 target leakage에 해당하지 않는다.
- 각 행의 관측값에 동일한 고정 규칙을 적용한 것이므로 test leakage에도 해당하지 않는다.
- 따라서 이 파생변수들은 모델이 원본 기록과 임상적 해석 신호를 함께 학습할 수 있도록 돕는 feature로 사용할 수 있다.

“이 변수들은 정답을 보고 만든 것이 아니라, 각 행에 이미 기록된 시술 정보로 만든 flag입니다. 예를 들어 해동된 배아가 1개 이상이면 냉동 배아 이식 가능성이 있다는 식의 고정 규칙을 적용한 것이므로 leakage가 아니라 feature engineering으로 볼 수 있습니다.”

### 02-5. 나이·난자·배아 구간화 변수 생성

EDA에서 나이, 난자 수, 배아 수는 단순 선형 관계보다 구간별 성공률 차이가 뚜렷하게 나타났다.  
따라서 원본값은 유지하면서 구간화 변수를 추가한다.


In [ ]:
# 나이 및 수량형 변수 구간화 함수 정의
age_ord_map = {
    "만18-34세": 0,
    "만35-37세": 1,
    "만38-39세": 2,
    "만40-42세": 3,
    "만43-44세": 4,
    "만45-50세": 5,
}


def make_oocyte_bin(series):
    x = pd.to_numeric(series, errors="coerce")
    result = pd.Series("missing", index=series.index, dtype="object")
    result[x == 0] = "0개"
    result[(x >= 1) & (x <= 5)] = "1-5개"
    result[(x >= 6) & (x <= 10)] = "6-10개"
    result[x >= 11] = "11개 이상"
    return result


def make_embryo_bin(series):
    x = pd.to_numeric(series, errors="coerce")
    result = pd.Series("missing", index=series.index, dtype="object")
    result[x == 0] = "0개"
    result[(x >= 1) & (x <= 2)] = "1-2개"
    result[(x >= 3) & (x <= 5)] = "3-5개"
    result[x >= 6] = "6개 이상"
    return result


def make_transfer_embryo_bin(series):
    x = pd.to_numeric(series, errors="coerce")
    result = pd.Series("missing", index=series.index, dtype="object")
    result[x == 0] = "0개"
    result[(x >= 1) & (x <= 2)] = "1-2개"
    result[x >= 3] = "3개 이상"
    return result


def add_binning_features(df):
    df["age_ord"] = df["시술 당시 나이"].map(age_ord_map)
    df["age_38plus_flag"] = (df["age_ord"] >= 2).fillna(False).astype(int)
    df["age_40plus_flag"] = (df["age_ord"] >= 3).fillna(False).astype(int)
    df["age_43plus_flag"] = (df["age_ord"] >= 4).fillna(False).astype(int)
    df["age_unknown_flag"] = df["age_ord"].isna().astype(int)

    oocyte_cols = ["수집된 신선 난자 수", "혼합된 난자 수", "해동 난자 수"]
    for col in oocyte_cols:
        if col in df.columns:
            df[col + "_bin"] = make_oocyte_bin(df[col])

    embryo_cols = ["총 생성 배아 수", "저장된 배아 수"]
    for col in embryo_cols:
        if col in df.columns:
            df[col + "_bin"] = make_embryo_bin(df[col])

    if "이식된 배아 수" in df.columns:
        df["이식된 배아 수_bin"] = make_transfer_embryo_bin(df["이식된 배아 수"])

    return df


X = add_binning_features(X)
X_test = add_binning_features(X_test)

bin_feature_cols = [col for col in X.columns if col.endswith("_bin")]
age_feature_cols = ["age_ord", "age_38plus_flag", "age_40plus_flag", "age_43plus_flag", "age_unknown_flag"]

print("생성된 구간화 변수 수:", len(bin_feature_cols))
print("나이 파생 변수:", age_feature_cols)
print("구간화 변수:", bin_feature_cols)

display(X[age_feature_cols + bin_feature_cols].head())

#### 02-5. 결과 해석

- `시술 당시 나이`를 순서형 변수인 `age_ord`로 변환하였다.
- `만18-34세`는 0, `만35-37세`는 1, `만38-39세`는 2처럼 나이대가 높아질수록 큰 값을 갖도록 설정하였다.
- EDA에서 성공률 하락이 뚜렷했던 기준을 반영하여 `age_38plus_flag`, `age_40plus_flag`, `age_43plus_flag`를 생성하였다.
- 나이 정보가 `알 수 없음`인 경우를 구분하기 위해 `age_unknown_flag`도 추가하였다.
- 난자 수 관련 변수는 `0개`, `1-5개`, `6-10개`, `11개 이상`으로 구간화하였다.
- 배아 수 관련 변수는 `0개`, `1-2개`, `3-5개`, `6개 이상`으로 구간화하였다.
- 이식된 배아 수는 `0개`, `1-2개`, `3개 이상`으로 구간화하였다.
- 원본 수치형 변수는 삭제하지 않고 유지하며, 구간화 변수는 비선형 성공률 패턴을 보완하는 feature로 사용한다.

### 02-6. 조합 변수 및 비율 변수 생성

범주형 변수는 단독으로도 의미가 있지만, 의료 데이터에서는 변수 간 조합이 더 중요한 신호가 될 수 있다.  
또한 난자·배아 관련 수량형 변수는 단순 개수뿐 아니라 비율 정보도 함께 활용한다.


In [ ]:
# 조합 변수 및 비율 변수 생성
def safe_ratio(numerator, denominator):
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")
    result = num / den.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan)


def add_combo_ratio_features(df):
    df["age_x_egg_source"] = (
        df["시술 당시 나이"].astype("string").fillna("missing")
        + "__"
        + df["난자 출처"].astype("string").fillna("missing")
    )

    df["treatment_x_specific_proc"] = (
        df["시술 유형"].astype("string").fillna("missing")
        + "__"
        + df["특정 시술 유형"].astype("string").fillna("missing")
    )

    df["age_x_specific_proc"] = (
        df["시술 당시 나이"].astype("string").fillna("missing")
        + "__"
        + df["특정 시술 유형"].astype("string").fillna("missing")
    )

    df["reason_x_specific_proc"] = (
        df["배아 생성 주요 이유"].astype("string").fillna("missing")
        + "__"
        + df["특정 시술 유형"].astype("string").fillna("missing")
    )

    df["transfer_per_created_embryo"] = safe_ratio(df["이식된 배아 수"], df["총 생성 배아 수"])
    df["stored_per_created_embryo"] = safe_ratio(df["저장된 배아 수"], df["총 생성 배아 수"])
    df["thawed_per_stored_embryo"] = safe_ratio(df["해동된 배아 수"], df["저장된 배아 수"])
    df["mixed_per_fresh_oocyte"] = safe_ratio(df["혼합된 난자 수"], df["수집된 신선 난자 수"])
    df["icsi_oocyte_rate"] = safe_ratio(df["미세주입된 난자 수"], df["혼합된 난자 수"])

    return df


X = add_combo_ratio_features(X)
X_test = add_combo_ratio_features(X_test)

combo_cols = [
    "age_x_egg_source",
    "treatment_x_specific_proc",
    "age_x_specific_proc",
    "reason_x_specific_proc",
]

ratio_cols = [
    "transfer_per_created_embryo",
    "stored_per_created_embryo",
    "thawed_per_stored_embryo",
    "mixed_per_fresh_oocyte",
    "icsi_oocyte_rate",
]

print("조합 변수 수:", len(combo_cols))
print("비율 변수 수:", len(ratio_cols))

display(X[combo_cols].head())
display(X[ratio_cols].describe())

#### 02-6. 결과 해석

- 주요 범주형 변수 간 조합 변수 4개와 배아·난자 관련 비율 변수 5개를 생성하였다.
- `age_x_egg_source`는 시술 당시 나이와 난자 출처의 조합을 나타낸다.
- `treatment_x_specific_proc`는 시술 유형과 특정 시술 유형의 조합을 나타낸다.
- `age_x_specific_proc`는 나이와 특정 시술 유형의 조합을 나타낸다.
- `reason_x_specific_proc`는 배아 생성 주요 이유와 특정 시술 유형의 조합을 나타낸다.
- 이러한 조합 변수는 단일 변수만으로는 포착하기 어려운 조건별 차이를 모델이 학습할 수 있도록 돕는다.

- 비율 변수에서는 `transfer_per_created_embryo`, `stored_per_created_embryo`, `thawed_per_stored_embryo`, `mixed_per_fresh_oocyte`, `icsi_oocyte_rate`를 생성하였다.
- `transfer_per_created_embryo`의 평균은 약 0.37로, 생성된 배아 중 이식된 배아의 비율을 나타낸다.
- `stored_per_created_embryo`의 평균은 약 0.18로, 생성된 배아 중 저장된 배아의 비율을 나타낸다.
- `mixed_per_fresh_oocyte`는 중앙값이 1.0으로, 많은 사례에서 수집된 신선 난자 수와 혼합된 난자 수가 유사하게 나타났다.
- `icsi_oocyte_rate`는 중앙값이 1.0으로, 미세주입된 난자 수가 혼합된 난자 수와 일치하는 사례가 많음을 보여준다.
- 일부 비율 변수의 최대값이 1을 초과하는데, 이는 분모가 작거나 기록 구조상 특정 수량 간 관계가 단순 비율로만 해석되기 어렵기 때문이다.
- 따라서 비율 변수는 임상적 의미를 참고하되, 최종 사용 여부는 모델 성능과 Feature Importance를 기준으로 판단한다.

### 02-7. 범주형/수치형 변수 정리

모델 입력에 사용할 feature에서 ID 컬럼을 제거하고, 범주형 변수와 수치형 변수를 구분한다.  

In [ ]:
# 모델 입력 feature 정리
X_model = X.drop(columns=[ID_COL], errors="ignore").copy()
X_test_model = X_test.drop(columns=[ID_COL], errors="ignore").copy()

assert TARGET not in X_model.columns
assert TARGET not in X_test_model.columns
assert list(X_model.columns) == list(X_test_model.columns)

cat_cols = X_model.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = [col for col in X_model.columns if col not in cat_cols]

print("X_model:", X_model.shape)
print("X_test_model:", X_test_model.shape)
print("전체 feature 수:", X_model.shape[1])
print("범주형 변수 수:", len(cat_cols))
print("수치형 변수 수:", len(num_cols))

print("\n범주형 변수 예시:")
print(cat_cols[:20])

print("\n수치형 변수 예시:")
print(num_cols[:20])


#### 02-7. 결과 해석

- ID 컬럼을 제외한 모델 입력 feature set을 구성하였다.
- 최종 feature 수는 133개로 정리되었다.
- 범주형 변수는 30개, 수치형 변수는 103개로 구분되었다.
- 범주형 변수에는 `시술 시기 코드`, `시술 당시 나이`, `시술 유형`, `특정 시술 유형`, `배란 유도 유형`, `배아 생성 주요 이유`, 시술 횟수 관련 변수, 난자/정자 출처 등이 포함된다.
- 수치형 변수에는 불임 원인 관련 이진 변수, 배아·난자 수량 변수, 경과일 변수, missing flag, clipping 변수, 임상 지식 기반 flag, 비율 변수 등이 포함된다.
- train과 제출용 데이터의 feature 수와 컬럼 순서가 동일함을 확인하였다.
- 이후 모델별 특성에 따라 범주형 변수는 문자열 유지 또는 인코딩을 적용하고, 수치형 변수는 필요에 따라 결측 대체 및 스케일링을 적용한다.

### 02-8. 인코딩 전략

범주형 변수는 모델별로 처리 방식이 다르다.  
CatBoost는 범주형 변수를 직접 처리할 수 있으므로 문자열 형태를 유지하고, LightGBM/XGBoost/Decision Tree/Random Forest 등은 숫자형 입력이 필요하므로 별도 인코딩을 적용한다.

In [ ]:
# CatBoost용 데이터: 범주형 변수 문자열 처리
X_cat = X_model.copy()
X_test_cat = X_test_model.copy()

for col in cat_cols:
    X_cat[col] = X_cat[col].astype("string").fillna("missing").astype(str)
    X_test_cat[col] = X_test_cat[col].astype("string").fillna("missing").astype(str)

cat_features = cat_cols.copy()

print("CatBoost용 X_cat:", X_cat.shape)
print("CatBoost용 X_test_cat:", X_test_cat.shape)
print("CatBoost cat_features 수:", len(cat_features))

In [ ]:
# LightGBM/XGBoost 및 트리 모델용 데이터: train 기준 Ordinal Encoding
from sklearn.preprocessing import OrdinalEncoder

X_ordinal = X_model.copy()
X_test_ordinal = X_test_model.copy()

ordinal_encoder = None

if len(cat_cols) > 0:
    ordinal_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    X_cat_values = X_ordinal[cat_cols].astype("string").fillna("missing")
    X_test_cat_values = X_test_ordinal[cat_cols].astype("string").fillna("missing")

    X_ordinal[cat_cols] = ordinal_encoder.fit_transform(X_cat_values)
    X_test_ordinal[cat_cols] = ordinal_encoder.transform(X_test_cat_values)

print("Ordinal Encoding 적용 완료")
print("X_ordinal:", X_ordinal.shape)
print("X_test_ordinal:", X_test_ordinal.shape)

#### 02-8. 결과 해석

- CatBoost용 데이터는 `X_cat`, 제출용 데이터는 `X_test_cat`으로 구성하였다.
- CatBoost는 범주형 변수를 직접 처리할 수 있으므로, 범주형 변수 30개를 문자열 형태로 유지하였다.
- CatBoost에서 사용할 범주형 변수 목록은 `cat_features`로 저장하였다.
- Decision Tree, Random Forest, LightGBM, XGBoost 등 숫자형 입력이 필요한 모델을 위해 `X_ordinal`, `X_test_ordinal`을 별도로 생성하였다.
- 범주형 변수는 train 데이터 기준으로 Ordinal Encoding을 적용하였다.
- 제출용 데이터에 train에서 보지 못한 새로운 범주가 나타날 경우 `-1`로 처리되도록 설정하였다.
- 여기서 `-1`은 수치형 이상치 처리가 아니라, 학습 데이터에서 보지 못한 새로운 범주를 의미하는 인코딩 값이다.
- 최종적으로 CatBoost용 데이터와 Ordinal Encoding 데이터 모두 133개 feature를 유지하였다.

### 02-9. 스케일링 전략

트리 기반 모델은 변수 스케일에 민감하지 않으므로 스케일링이 필수는 아니다.  
반면 Logistic Regression, MLP 등 선형·신경망 계열 모델은 결측 대체와 스케일링이 필요하다.

따라서 스케일링은 별도의 선형/MLP용 데이터셋에만 적용한다.


In [ ]:
# 선형/MLP 모델용 결측 대체 및 스케일링
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X_linear_base = X_ordinal.copy()
X_test_linear_base = X_test_ordinal.copy()

num_like_cols = X_linear_base.columns.tolist()

linear_imputer = SimpleImputer(strategy="median")
linear_scaler = StandardScaler()

X_linear_imputed = pd.DataFrame(
    linear_imputer.fit_transform(X_linear_base),
    columns=num_like_cols,
    index=X_linear_base.index,
)

X_test_linear_imputed = pd.DataFrame(
    linear_imputer.transform(X_test_linear_base),
    columns=num_like_cols,
    index=X_test_linear_base.index,
)

X_linear_scaled = pd.DataFrame(
    linear_scaler.fit_transform(X_linear_imputed),
    columns=num_like_cols,
    index=X_linear_imputed.index,
)

X_test_linear_scaled = pd.DataFrame(
    linear_scaler.transform(X_test_linear_imputed),
    columns=num_like_cols,
    index=X_test_linear_imputed.index,
)

print("X_linear_scaled:", X_linear_scaled.shape)
print("X_test_linear_scaled:", X_test_linear_scaled.shape)

#### 02-9. 결과 해석

- Logistic Regression, MLP 등 선형·신경망 계열 모델을 위한 별도 데이터셋을 생성하였다.
- 결측값은 train 데이터의 중앙값 기준으로 대체하였다.
- 이후 StandardScaler를 적용하여 각 feature의 스케일을 표준화하였다.
- 스케일링 결과 `X_linear_scaled`는 256,351행 133개 feature로 구성되었다.
- 제출용 데이터에 동일한 변환을 적용한 `X_test_linear_scaled`는 90,067행 133개 feature로 구성되었다.
- CatBoost, Decision Tree, Random Forest, LightGBM, XGBoost와 같은 트리 기반 모델은 스케일링 없이 사용할 수 있다.

### 02-10. 학습/검증 데이터 분리

모델 성능을 검증하기 위해 train 데이터를 학습용 데이터와 검증용 데이터로 분리한다.  
임신 성공 여부는 클래스 불균형이 존재하므로, target 비율이 유지되도록 stratify를 적용한다.

In [ ]:
# 학습/검증 데이터 분리 및 모델별 입력 데이터 준비
from sklearn.model_selection import train_test_split, StratifiedKFold

train_idx, valid_idx = train_test_split(
    np.arange(len(X_model)),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_cb = X_cat.iloc[train_idx].copy()
X_valid_cb = X_cat.iloc[valid_idx].copy()
X_test_cb = X_test_cat.copy()

X_train_gbdt = X_ordinal.iloc[train_idx].copy()
X_valid_gbdt = X_ordinal.iloc[valid_idx].copy()
X_test_gbdt = X_test_ordinal.copy()

X_train_linear = X_linear_scaled.iloc[train_idx].copy()
X_valid_linear = X_linear_scaled.iloc[valid_idx].copy()
X_test_linear = X_test_linear_scaled.copy()

y_train = y.iloc[train_idx].copy()
y_valid = y.iloc[valid_idx].copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("X_train_cb:", X_train_cb.shape)
print("X_valid_cb:", X_valid_cb.shape)
print("X_test_cb:", X_test_cb.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

print("\nTrain target ratio:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nValid target ratio:")
print(y_valid.value_counts(normalize=True).sort_index())

#### 02-10. 결과 해석

- train 데이터를 학습용 데이터와 검증용 데이터로 분리하였다.
- 학습 데이터는 205,080건, 검증 데이터는 51,271건으로 구성되었다.
- CatBoost용 데이터 기준 feature 수는 133개로 유지되었다.
- 제출용 데이터는 90,067건이며, 동일한 133개 feature 구조를 가진다.
- 클래스 불균형을 고려하여 `stratify=y`를 적용하였다.
- 그 결과 학습 데이터의 성공 비율은 약 25.83%, 검증 데이터의 성공 비율도 약 25.84%로 거의 동일하게 유지되었다.
- 따라서 검증 데이터는 전체 train 데이터의 target 분포를 잘 대표한다고 볼 수 있다.
- 이후 모델링 단계에서는 동일한 train/valid 분할을 기준으로 모델 성능을 비교한다.

## 03. 모델링

전처리된 feature를 활용하여 임신 성공 여부를 예측하는 분류 모델을 학습한다.  
먼저 baseline 모델을 통해 기본 성능을 확인하고, 이후 CatBoost, LightGBM, XGBoost 등 고성능 트리 기반 모델과 비교한다.

### 03-1. Baseline 모델 성능 비교

로지스틱 회귀, 결정트리, 랜덤포레스트를 baseline 모델로 학습한다.  
각 모델의 성능은 ROC-AUC, LogLoss, Accuracy, Precision, Recall, F1-score를 기준으로 비교한다.

In [ ]:
# 모델 평가 함수 정의
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

def evaluate_binary_model(model_name, y_true, pred_proba, threshold=0.5):
    pred_label = (pred_proba >= threshold).astype(int)

    return {
        "model": model_name,
        "ROC-AUC": roc_auc_score(y_true, pred_proba),
        "LogLoss": log_loss(y_true, pred_proba),
        "Accuracy": accuracy_score(y_true, pred_label),
        "Precision": precision_score(y_true, pred_label, zero_division=0),
        "Recall": recall_score(y_true, pred_label, zero_division=0),
        "F1-score": f1_score(y_true, pred_label, zero_division=0),
    }

In [ ]:
# Baseline 모델 학습 및 평가
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

baseline_results = []

# Logistic Regression
logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

logistic_model.fit(X_train_linear, y_train)
logistic_valid_proba = logistic_model.predict_proba(X_valid_linear)[:, 1]

baseline_results.append(
    evaluate_binary_model(
        "Logistic Regression",
        y_valid,
        logistic_valid_proba
    )
)

# Decision Tree
tree_model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

tree_model.fit(X_train_gbdt, y_train)
tree_valid_proba = tree_model.predict_proba(X_valid_gbdt)[:, 1]

baseline_results.append(
    evaluate_binary_model(
        "Decision Tree",
        y_valid,
        tree_valid_proba
    )
)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=30,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_model.fit(X_train_gbdt, y_train)
rf_valid_proba = rf_model.predict_proba(X_valid_gbdt)[:, 1]

baseline_results.append(
    evaluate_binary_model(
        "Random Forest",
        y_valid,
        rf_valid_proba
    )
)

baseline_results_df = pd.DataFrame(baseline_results).set_index("model")
baseline_results_df

#### 03-1. 결과 해석

- Baseline 모델 3개를 비교한 결과, ROC-AUC 기준으로 Random Forest가 0.7317로 가장 높았다.
- Logistic Regression의 ROC-AUC는 0.7284, Decision Tree는 0.7279로 나타났다.
- 세 모델 모두 ROC-AUC가 약 0.73 수준으로, 기본 전처리와 파생변수만으로도 일정 수준의 예측력이 확보되었다.
- LogLoss는 Random Forest가 0.5890으로 가장 낮아, 예측 확률의 품질도 세 모델 중 가장 안정적이었다.
- Decision Tree는 Recall이 0.7926으로 가장 높았지만, Precision은 0.3723으로 가장 낮았다.
- 이는 Decision Tree가 성공 클래스를 더 많이 잡아내는 대신, 성공으로 잘못 예측하는 경우도 상대적으로 많다는 의미이다.
- Logistic Regression과 Random Forest는 Accuracy, Precision, Recall, F1-score가 매우 유사하게 나타났다.
- Random Forest는 여러 결정트리를 결합하는 방식이기 때문에 단일 Decision Tree보다 예측 안정성이 높고, 실제 결과에서도 ROC-AUC와 LogLoss가 개선되었다.
- 이후 모델링에서는 Random Forest를 baseline 기준점으로 삼고, CatBoost, LightGBM, XGBoost와 같은 boosting 계열 모델이 이를 얼마나 개선하는지 확인한다.

### 03-2. CatBoost 모델 학습

CatBoost는 범주형 변수를 직접 처리할 수 있는 Gradient Boosting 계열 모델이다.  
본 데이터는 범주형 변수가 많고 결측값 자체가 시술 프로세스의 의미를 가질 수 있으므로, CatBoost를 주요 후보 모델로 학습한다.

In [ ]:
# CatBoost 모델 학습
from catboost import CatBoostClassifier, Pool

cat_train_pool = Pool(
    X_train_cb,
    y_train,
    cat_features=cat_features,
)

cat_valid_pool = Pool(
    X_valid_cb,
    y_valid,
    cat_features=cat_features,
)

cat_model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    auto_class_weights="Balanced",
    early_stopping_rounds=200,
    verbose=200,
)

cat_model.fit(
    cat_train_pool,
    eval_set=cat_valid_pool,
    use_best_model=True,
)

cat_valid_proba = cat_model.predict_proba(cat_valid_pool)[:, 1]

cat_metrics = evaluate_binary_model(
    "CatBoost",
    y_valid,
    cat_valid_proba
)

cat_metrics

#### CatBoost 학습 로그 해석

- CatBoost는 최대 3,000 iteration까지 학습하도록 설정했지만, early stopping이 적용되어 1,838번째 부근에서 학습이 중단되었다.
- 가장 좋은 검증 ROC-AUC는 1,638번째 iteration에서 0.73735로 나타났다.
- 이후 200번의 iteration 동안 검증 성능이 더 이상 개선되지 않아 overfitting detector가 작동하였다.
- 최종 모델은 가장 성능이 좋았던 1,639개 tree까지만 사용하도록 축소되었다.
- 이는 iteration을 더 늘리면 train 데이터에는 더 맞춰질 수 있지만, 검증 성능은 개선되지 않는다는 의미이다.

In [ ]:
cat_valid_proba = cat_model.predict_proba(cat_valid_pool)[:, 1]

cat_metrics = evaluate_binary_model(
    "CatBoost",
    y_valid,
    cat_valid_proba
)

model_results_df = pd.concat([
    baseline_results_df,
    pd.DataFrame([cat_metrics]).set_index("model")
])

model_results_df

#### 03-2. 결과 해석

- CatBoost의 ROC-AUC는 0.7374로, baseline 모델 중 가장 높았던 Random Forest의 0.7317보다 개선되었다.
- LogLoss도 CatBoost가 0.5873으로 가장 낮아, 예측 확률의 품질이 baseline 모델보다 개선되었다.
- Accuracy는 0.6217로 Random Forest의 0.6192보다 소폭 높았다.
- Precision은 0.3850, Recall은 0.7770, F1-score는 0.5148로 나타났다.
- 특히 Recall과 F1-score가 baseline 모델보다 개선되어, 성공 클래스를 더 잘 포착하는 방향으로 성능이 향상되었다.
- CatBoost는 범주형 변수를 직접 처리하고, 결측값과 범주형 조합을 효과적으로 학습할 수 있어 본 데이터에 적합한 모델로 판단된다.
- 따라서 이후 모델링에서는 CatBoost를 주요 기준 모델로 삼고, LightGBM, XGBoost와 성능을 비교한다.

### 03-3. LightGBM 모델 학습

LightGBM은 Gradient Boosting Decision Tree 계열 모델로, 학습 속도가 빠르고 tabular 데이터에서 강한 성능을 보이는 모델이다.  
범주형 변수는 앞서 Ordinal Encoding한 데이터를 사용하고, 결측값은 LightGBM이 자체적으로 처리하도록 둔다.

In [ ]:
# LightGBM 모델 학습
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgbm_model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

lgbm_model.fit(
    X_train_gbdt,
    y_train,
    eval_set=[(X_valid_gbdt, y_valid)],
    eval_metric="auc",
    callbacks=[
        early_stopping(stopping_rounds=200),
        log_evaluation(period=200),
    ],
)

lgbm_valid_proba = lgbm_model.predict_proba(X_valid_gbdt)[:, 1]

lgbm_metrics = evaluate_binary_model(
    "LightGBM",
    y_valid,
    lgbm_valid_proba
)

lgbm_metrics

In [ ]:
# 기존 결과표에 LightGBM 결과 추가
model_results_df = pd.concat([
    model_results_df,
    pd.DataFrame([lgbm_metrics]).set_index("model")
])

model_results_df

#### 03-3. 결과 해석

- LightGBM의 ROC-AUC는 0.7366으로 나타났다.
- 이는 baseline 모델 중 가장 높았던 Random Forest의 0.7317보다 높은 성능이다.
- LogLoss는 0.5876으로 Random Forest의 0.5890보다 낮아, 예측 확률의 품질도 개선되었다.
- Recall은 0.7795로 CatBoost보다도 약간 높게 나타났다.
- 다만 Precision은 0.3842, F1-score는 0.5147로 CatBoost와 거의 유사한 수준이다.
- CatBoost와 비교하면 LightGBM은 Recall은 소폭 높지만, ROC-AUC와 LogLoss는 CatBoost가 조금 더 우수하다.
- 따라서 현재 단일 모델 기준으로는 CatBoost가 가장 우수하지만, LightGBM도 성능이 비슷하므로 이후 ensemble 후보로 활용할 수 있다.

### 03-4. XGBoost 모델 학습

XGBoost는 Gradient Boosting Decision Tree 계열의 대표적인 모델이다.  
LightGBM, CatBoost와 함께 tabular 데이터에서 자주 사용되며, 모델 간 성능 비교 및 ensemble 후보로 활용할 수 있다.

In [ ]:
# XGBoost 모델 학습
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
)

xgb_model.fit(
    X_train_gbdt,
    y_train,
    eval_set=[(X_valid_gbdt, y_valid)],
    verbose=200,
)

xgb_valid_proba = xgb_model.predict_proba(X_valid_gbdt)[:, 1]

xgb_metrics = evaluate_binary_model(
    "XGBoost",
    y_valid,
    xgb_valid_proba
)

xgb_metrics

In [ ]:
model_results_df = pd.concat([
    model_results_df,
    pd.DataFrame([xgb_metrics]).set_index("model")
])

model_results_df

#### 03-4. 결과 해석

- XGBoost의 ROC-AUC는 0.7330으로 Random Forest보다는 높았지만, CatBoost와 LightGBM보다는 낮게 나타났다.
- XGBoost의 LogLoss는 0.5855로 전체 모델 중 가장 낮았다.
- 이는 XGBoost의 예측 확률 품질이 상대적으로 안정적일 수 있음을 의미한다.
- Accuracy와 Precision도 XGBoost가 가장 높게 나타났지만, Recall은 0.7551로 CatBoost와 LightGBM보다 낮았다.
- 즉, XGBoost는 성공 클래스를 더 보수적으로 예측하는 경향이 있으며, 성공을 많이 잡아내는 능력은 CatBoost/LightGBM보다 낮았다.
- ROC-AUC 기준으로는 CatBoost가 0.7374로 가장 우수했고, LightGBM이 0.7366으로 그 뒤를 이었다.
- 따라서 단일 모델 기준의 핵심 후보는 CatBoost이며, LightGBM은 성능이 유사하므로 ensemble 후보로 활용할 수 있다.
- XGBoost는 ROC-AUC는 낮지만 LogLoss가 가장 낮아, 확률 보정이나 ensemble에서 보완적 역할을 할 가능성이 있다.

#### 단일 모델 비교 종합

- Baseline 모델 중에서는 Random Forest가 ROC-AUC 0.7317로 가장 우수했다.
- Boosting 계열 모델은 전반적으로 baseline 모델보다 높은 ROC-AUC를 보였다.
- CatBoost는 ROC-AUC 0.7374로 가장 높은 순위 예측력을 보였다.
- LightGBM은 ROC-AUC 0.7366으로 CatBoost와 유사한 성능을 보였다.
- XGBoost는 ROC-AUC는 0.7330으로 낮았지만, LogLoss 0.5855로 가장 낮은 값을 기록하였다.
- 최종적으로 CatBoost를 주요 단일 모델로 보고, CatBoost·LightGBM·XGBoost의 예측값을 조합하는 ensemble을 검토한다.

### 03-5. 모델 성능 비교 시각화

모델별 성능 차이를 직관적으로 확인하기 위해 ROC-AUC와 LogLoss를 시각화한다.  
ROC-AUC는 높을수록 좋고, LogLoss는 낮을수록 좋은 지표이다.

In [ ]:
# 모델 성능 비교 시각화용 데이터 정리
model_plot_df = model_results_df.reset_index().copy()

display(model_plot_df)

In [ ]:
# ROC-AUC 비교 시각화
plot_df = model_plot_df.sort_values("ROC-AUC", ascending=True)

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=plot_df,
    x="ROC-AUC",
    y="model",
    color=main_color
)

for i, row in plot_df.iterrows():
    ax.text(
        row["ROC-AUC"] + 0.001,
        list(plot_df.index).index(i),
        f'{row["ROC-AUC"]:.4f}',
        va="center",
        fontsize=10
    )

ax.set_title("모델별 ROC-AUC 비교")
ax.set_xlabel("ROC-AUC")
ax.set_ylabel("")
ax.set_xlim(plot_df["ROC-AUC"].min() - 0.005, plot_df["ROC-AUC"].max() + 0.01)

plt.tight_layout()
plt.show()

In [ ]:
# LogLoss 비교 시각화
plot_df = model_plot_df.sort_values("LogLoss", ascending=False)

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=plot_df,
    x="LogLoss",
    y="model",
    color=accent_color
)

for i, row in plot_df.iterrows():
    ax.text(
        row["LogLoss"] + 0.001,
        list(plot_df.index).index(i),
        f'{row["LogLoss"]:.4f}',
        va="center",
        fontsize=10
    )

ax.set_title("모델별 LogLoss 비교")
ax.set_xlabel("LogLoss")
ax.set_ylabel("")
ax.set_xlim(plot_df["LogLoss"].min() - 0.005, plot_df["LogLoss"].max() + 0.01)

plt.tight_layout()
plt.show()

#### 03-5. 결과 해석

- ROC-AUC 기준으로는 CatBoost가 가장 높은 성능을 보였다.
- LightGBM은 CatBoost와 매우 유사한 ROC-AUC를 기록하였다.
- Baseline 모델 중에서는 Random Forest가 가장 우수하였다.
- LogLoss 기준으로는 XGBoost가 가장 낮은 값을 보였다.
- 이는 XGBoost가 ROC-AUC는 최고가 아니지만, 예측 확률의 안정성 측면에서는 장점이 있을 수 있음을 의미한다.
- 따라서 최종 모델링에서는 CatBoost를 주요 모델로 두고, LightGBM과 XGBoost를 ensemble 후보로 검토한다.

### 03-6. Feature Importance 확인

가장 높은 ROC-AUC를 보인 CatBoost 모델을 기준으로 feature importance를 확인한다.  
이를 통해 모델이 어떤 변수들을 주요 예측 신호로 활용했는지 파악하고, 이후 noise pruning 후보를 검토한다.

In [ ]:
# CatBoost Feature Importance 확인
cat_feature_importance = pd.DataFrame({
    "feature": X_train_cb.columns,
    "importance": cat_model.get_feature_importance(cat_valid_pool),
})

cat_feature_importance = cat_feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

display(cat_feature_importance.head(30))

In [ ]:
# 상위 Feature Importance 시각화
top_n = 25
plot_fi = cat_feature_importance.head(top_n).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 8))
ax = sns.barplot(
    data=plot_fi,
    x="importance",
    y="feature",
    color=main_color
)

for i, row in plot_fi.iterrows():
    ax.text(
        row["importance"] + 0.1,
        list(plot_fi.index).index(i),
        f'{row["importance"]:.2f}',
        va="center",
        fontsize=9
    )

ax.set_title("CatBoost Feature Importance Top 25")
ax.set_xlabel("Importance")
ax.set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# 하위 Feature Importance 확인
display(cat_feature_importance.tail(30))

#### 03-6. Feature Importance 결과 해석

- CatBoost 기준 Feature Importance를 확인한 결과, 상위 변수는 배아 이식, 나이, 난자 출처, 배아 생성·저장 관련 변수에 집중되어 있었다.
- 가장 중요한 변수는 `이식된 배아 수`로 importance가 27.52로 나타났다.
- 두 번째로 중요한 변수는 `이식된 배아 수_bin`으로 importance가 26.35였다.
- 이는 이식된 배아 수 자체와 그 구간 정보가 임신 성공 여부 예측에서 핵심적인 역할을 한다는 의미이다.
- 세 번째로 중요한 변수는 `age_x_egg_source`로 importance가 10.46이었다.
- 이는 나이와 난자 출처의 조합이 단일 변수보다 중요한 예측 신호가 될 수 있음을 보여준다.
- `총 생성 배아 수_bin`, `배아 이식 경과일`, `stored_per_created_embryo`, `배아 이식 경과일_is_missing`도 상위권에 포함되었다.
- 즉, 모델은 단순한 환자 정보보다 실제 배아 생성·이식 과정과 관련된 변수를 주요 신호로 활용하고 있다.
- `시술 당시 나이`, `age_ord`, `age_38plus_flag`도 상위 변수에 포함되어, EDA에서 확인한 나이 효과가 모델에서도 반영되었다.
- clipping 변수 중 `저장된 배아 수_clipped`, `혼합된 난자 수_clipped`, `총 생성 배아 수_clipped`가 상위권에 일부 포함되어, 이상치 보정 후보 변수도 일정한 정보를 제공하는 것으로 보인다.

- 하위 Feature Importance에는 일부 결측 flag와 희소한 이진 변수가 많이 포함되었다.
- `process_missing_count`와 여러 구조적 missing flag는 중요도가 0으로 나타났다.
- 이는 구조적 결측 정보가 이미 `시술 유형`, 원본 변수의 NaN, 또는 다른 관련 변수에 의해 충분히 설명되고 있기 때문으로 해석할 수 있다.
- `총 생성 배아 수_missing_flag`, `이식된 배아 수_missing_flag`, `미세주입된 난자 수_missing_flag` 등은 모두 DI 구조적 결측을 거의 동일하게 나타내므로 중복 정보일 가능성이 높다.
- `특정 시술 유형_is_missing`은 결측 건수가 매우 적어 모델에서 거의 활용되지 않았다.
- `unknown_proc_with_icsi_oocyte_flag` 역시 해당 조건이 74건으로 매우 적어 중요도가 낮게 나타난 것으로 보인다.
- `불임 원인 - 정자 면역학적 요인`, `불임 원인 - 여성 요인`, `불임 원인 - 정자 형태` 등 일부 불임 원인 변수도 중요도가 0으로 나타났다.
- 따라서 구조적 missing flag 중 중요도 0인 변수, 결측 건수가 매우 적은 변수, 발생 빈도가 낮은 flag, 일부 불임 원인 변수는 noise pruning 후보로 볼 수 있다.
- 다만 Feature Importance가 낮다고 해서 반드시 제거해야 하는 것은 아니다.
- 다른 모델이나 interaction, ensemble 환경에서는 낮은 중요도의 변수가 보조 신호로 작동할 수 있으므로, 제거 여부는 OOF 성능 변화를 확인하면서 판단한다.

### 03-7. Ensemble 후보 비교

CatBoost, LightGBM, XGBoost는 서로 다른 방식으로 데이터를 학습하므로 예측값을 조합하면 성능이 개선될 수 있다.  
검증 데이터에서 각 모델의 예측 확률을 가중 평균하여 ROC-AUC와 LogLoss 변화를 확인한다.

In [ ]:
# Ensemble 후보 가중치 비교
ensemble_results = []

weight_grid = [
    (0.6, 0.3, 0.1),
    (0.5, 0.4, 0.1),
    (0.5, 0.3, 0.2),
    (0.4, 0.4, 0.2),
    (0.4, 0.3, 0.3),
    (0.7, 0.2, 0.1),
    (0.8, 0.1, 0.1),
]

for cat_w, lgbm_w, xgb_w in weight_grid:
    ensemble_proba = (
        cat_w * cat_valid_proba
        + lgbm_w * lgbm_valid_proba
        + xgb_w * xgb_valid_proba
    )

    metrics = evaluate_binary_model(
        f"Ensemble C{cat_w}_L{lgbm_w}_X{xgb_w}",
        y_valid,
        ensemble_proba
    )

    metrics["cat_weight"] = cat_w
    metrics["lgbm_weight"] = lgbm_w
    metrics["xgb_weight"] = xgb_w

    ensemble_results.append(metrics)

ensemble_results_df = pd.DataFrame(ensemble_results).set_index("model")
ensemble_results_df = ensemble_results_df.sort_values("ROC-AUC", ascending=False)

display(ensemble_results_df)

In [ ]:
# 단일 모델과 Ensemble 결과 비교
best_ensemble = ensemble_results_df.head(1)

model_compare_with_ensemble = pd.concat([
    model_results_df,
    best_ensemble.drop(columns=["cat_weight", "lgbm_weight", "xgb_weight"])
])

display(model_compare_with_ensemble.sort_values("ROC-AUC", ascending=False))

#### 03-7. 결과 해석

- CatBoost, LightGBM, XGBoost의 예측 확률을 가중 평균하여 ensemble 후보를 비교하였다.
- 검증셋 기준 가장 높은 ROC-AUC는 `CatBoost 0.7 + LightGBM 0.2 + XGBoost 0.1` 조합에서 나타났다.
- 해당 ensemble의 ROC-AUC는 0.7375로, 단일 CatBoost의 0.7374보다 소폭 개선되었다.
- LogLoss는 0.5865로 CatBoost의 0.5873보다 낮아져, 예측 확률의 품질도 일부 개선되었다.
- Accuracy와 F1-score 역시 단일 CatBoost보다 약간 높게 나타났다.
- 다만 개선 폭은 크지 않으므로, 단일 검증셋에서의 가중치 최적화 결과만으로 최종 성능 향상을 확정하기는 어렵다.
- XGBoost 비중이 커질수록 LogLoss는 낮아지는 경향이 있었지만, ROC-AUC는 소폭 하락하였다.
- 이는 XGBoost가 확률 품질에는 기여할 수 있으나, 순위 예측력 측면에서는 CatBoost/LightGBM보다 약한 신호를 보였기 때문으로 해석할 수 있다.
- 최종 제출 모델에서는 CatBoost 중심의 ensemble을 우선 후보로 두되, K-Fold OOF 기반 검증을 통해 가중치의 안정성을 추가로 확인할 필요가 있다.

### 03-8. K-Fold OOF 검증: CatBoost

단일 train/valid split은 데이터 분할에 따라 성능이 달라질 수 있다.  
따라서 Stratified K-Fold를 사용하여 전체 train 데이터를 여러 fold로 나누고, 각 fold의 validation 예측을 모아 OOF 성능을 계산한다.

OOF 예측값은 각 행이 학습에 사용되지 않았던 fold에서 얻은 예측값이므로, 단일 검증셋보다 더 안정적인 내부 검증 지표로 활용할 수 있다.

In [ ]:
# CatBoost 5-Fold OOF 검증
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, log_loss

N_SPLITS = 5

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cat_oof_pred = np.zeros(len(X_cat))
cat_test_pred = np.zeros(len(X_test_cat))
cat_fold_results = []
cat_models = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cat, y), 1):
    print(f"\n========== Fold {fold} ==========")

    X_tr = X_cat.iloc[tr_idx].copy()
    X_val = X_cat.iloc[val_idx].copy()
    y_tr = y.iloc[tr_idx].copy()
    y_val = y.iloc[val_idx].copy()

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    valid_pool = Pool(X_val, y_val, cat_features=cat_features)
    test_pool = Pool(X_test_cat, cat_features=cat_features)

    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE + fold,
        auto_class_weights="Balanced",
        early_stopping_rounds=200,
        verbose=200,
    )

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
    )

    val_pred = model.predict_proba(valid_pool)[:, 1]
    test_pred = model.predict_proba(test_pool)[:, 1]

    cat_oof_pred[val_idx] = val_pred
    cat_test_pred += test_pred / N_SPLITS
    cat_models.append(model)

    fold_auc = roc_auc_score(y_val, val_pred)
    fold_logloss = log_loss(y_val, val_pred)

    cat_fold_results.append({
        "fold": fold,
        "ROC-AUC": fold_auc,
        "LogLoss": fold_logloss,
        "best_iteration": model.get_best_iteration(),
    })

    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Fold {fold} LogLoss: {fold_logloss:.6f}")

cat_fold_results_df = pd.DataFrame(cat_fold_results)

cat_oof_metrics = evaluate_binary_model(
    "CatBoost 5-Fold OOF",
    y,
    cat_oof_pred
)

display(cat_fold_results_df)
display(pd.DataFrame([cat_oof_metrics]).set_index("model"))

#### 03-8. CatBoost 5-Fold OOF 결과 해석

5개 fold의 ROC-AUC는 0.738410 ~ 0.742708 범위로 나타났다.
평균적으로는 약 0.74026 수준이며, 최종 OOF ROC-AUC도 0.740262로 거의 일치한다.
단일 검증 분할 CatBoost 결과인 0.737350보다 개선되었다.
Fold 2가 가장 높고, Fold 4가 가장 낮지만 fold 간 차이가 지나치게 크지는 않다.
따라서 현재 CatBoost 모델은 단일 split보다 K-Fold 기반 검증에서 더 안정적인 성능을 보였다고 해석할 수 있다.

### 03-9. LightGBM 5-Fold OOF

CatBoost 5-Fold 단일 모델의 제출 점수를 기준선으로 두고, LightGBM을 동일한 K-Fold 구조로 학습한다.  
이후 CatBoost와 LightGBM의 OOF 예측값을 조합하여 앙상블 성능 개선 가능성을 확인한다.

In [ ]:
# LightGBM 5-Fold OOF 학습
import lightgbm as lgb

lgb_oof_pred = np.zeros(len(X_ordinal))
lgb_test_pred = np.zeros(len(X_test_ordinal))
lgb_fold_results = []
lgb_models = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_ordinal, y), start=1):
    print(f"\n========== Fold {fold} ==========")

    X_tr = X_ordinal.iloc[train_idx]
    X_val = X_ordinal.iloc[valid_idx]
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[valid_idx]

    lgb_model = lgb.LGBMClassifier(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
        objective="binary"
    )

    lgb_model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=200)
        ]
    )

    val_pred = lgb_model.predict_proba(X_val)[:, 1]
    test_pred = lgb_model.predict_proba(X_test_ordinal)[:, 1]

    lgb_oof_pred[valid_idx] = val_pred
    lgb_test_pred += test_pred / skf.n_splits

    fold_auc = roc_auc_score(y_val, val_pred)
    fold_logloss = log_loss(y_val, val_pred)

    lgb_fold_results.append({
        "fold": fold,
        "ROC-AUC": fold_auc,
        "LogLoss": fold_logloss,
        "best_iteration": lgb_model.best_iteration_
    })

    lgb_models.append(lgb_model)

    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Fold {fold} LogLoss: {fold_logloss:.6f}")

lgb_fold_results_df = pd.DataFrame(lgb_fold_results)
display(lgb_fold_results_df)

lgb_oof_metrics = evaluate_binary_model("LightGBM 5-Fold OOF", y, lgb_oof_pred)
display(pd.DataFrame([lgb_oof_metrics]).set_index("model"))

#### 03-9. LightGBM 5-Fold OOF 결과 해석

LightGBM 5-Fold OOF의 최종 ROC-AUC는 0.739595로 나타났다.
기존 CatBoost 5-Fold OOF의 ROC-AUC 0.740262보다는 소폭 낮다.
Fold별 ROC-AUC는 0.737326 ~ 0.742176 범위로 나타났으며, Fold 2가 가장 높고 Fold 1이 가장 낮았다.
Fold 간 성능 차이는 존재하지만, 전체적으로 0.739~0.742 구간에 모여 있어 검증 성능이 크게 흔들리는 수준은 아니다.
LogLoss는 최종 0.586069로, CatBoost 5-Fold OOF의 0.585947과 거의 비슷하다.
Recall은 0.788156으로 CatBoost보다 약간 높아, 임신 성공 클래스를 더 적극적으로 예측하는 경향이 있다.
반면 ROC-AUC 기준으로는 CatBoost가 더 우수하므로, LightGBM 단독 모델보다는 CatBoost와의 앙상블 후보로 활용하는 것이 적절하다.
정리하면, LightGBM은 단독 1순위 모델은 아니지만 CatBoost와 예측 패턴이 다를 수 있어 OOF 앙상블에 포함할 가치가 있는 보조 모델로 판단된다.

### 03-10. CatBoost + LightGBM OOF 앙상블 가중치 탐색

In [ ]:
# CatBoost + LightGBM OOF 앙상블 가중치 탐색
ensemble_results = []

for cat_weight in np.arange(0.0, 1.01, 0.05):
    lgb_weight = 1 - cat_weight

    ensemble_oof_pred = (
        cat_weight * cat_oof_pred +
        lgb_weight * lgb_oof_pred
    )

    metrics = evaluate_binary_model(
        f"Ensemble C{cat_weight:.2f}_L{lgb_weight:.2f}",
        y,
        ensemble_oof_pred
    )

    metrics["cat_weight"] = cat_weight
    metrics["lgb_weight"] = lgb_weight

    ensemble_results.append(metrics)

ensemble_results_df = (
    pd.DataFrame(ensemble_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(ensemble_results_df)

#### 03-10. 결과 해석

최고 조합은 CatBoost 0.70 + LightGBM 0.30이다.
ROC-AUC는 0.740422로, CatBoost 단독 0.740262보다 소폭 상승했다.
상승폭은 약 +0.000160으로 크지는 않지만, 앙상블 효과는 확인되었다.
LogLoss는 0.585715로 CatBoost 단독 0.585947보다 개선되었다.
대회 평가지표가 ROC-AUC이므로 최종 제출 후보는 C0.70_L0.30이 가장 적절하다.

In [ ]:
# 03-11. 세밀 가중치 탐색
fine_ensemble_results = []

for cat_weight in np.arange(0.60, 0.801, 0.01):
    lgb_weight = 1 - cat_weight

    ensemble_oof_pred = (
        cat_weight * cat_oof_pred +
        lgb_weight * lgb_oof_pred
    )

    metrics = evaluate_binary_model(
        f"Fine Ensemble C{cat_weight:.2f}_L{lgb_weight:.2f}",
        y,
        ensemble_oof_pred
    )

    metrics["cat_weight"] = cat_weight
    metrics["lgb_weight"] = lgb_weight
    fine_ensemble_results.append(metrics)

fine_ensemble_results_df = (
    pd.DataFrame(fine_ensemble_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(fine_ensemble_results_df.head(10))

30퍼 근처(313233 보고 정밀 탐색) -> 결과 같음. 오히려 나머지 평가 지표는 떨어짐

In [ ]:
# 03-12. Rank Average 앙상블

from scipy.stats import rankdata

cat_oof_rank = rankdata(cat_oof_pred) / len(cat_oof_pred)
lgb_oof_rank = rankdata(lgb_oof_pred) / len(lgb_oof_pred)

rank_ensemble_results = []

for cat_weight in np.arange(0.60, 0.801, 0.01):
    lgb_weight = 1 - cat_weight

    rank_oof_pred = (
        cat_weight * cat_oof_rank +
        lgb_weight * lgb_oof_rank
    )

    metrics = evaluate_binary_model(
        f"Rank Ensemble C{cat_weight:.2f}_L{lgb_weight:.2f}",
        y,
        rank_oof_pred
    )

    metrics["cat_weight"] = cat_weight
    metrics["lgb_weight"] = lgb_weight
    rank_ensemble_results.append(metrics)

rank_ensemble_results_df = (
    pd.DataFrame(rank_ensemble_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(rank_ensemble_results_df.head(10))

### 03-13. AUC를 높일 수 있는 방향(개선안)

현재 v3의 흐름은 CatBoost와 LightGBM 5-Fold OOF를 만들고, 두 모델의 확률/순위 앙상블을 비교하는 구조이다.  
v5에서는 v4의 아이디어를 이어받아 다음 방향으로 AUC 개선을 시도한다.

- XGBoost도 5-Fold OOF로 학습하여 CatBoost/LightGBM과 같은 검증 구조로 맞춘다.
- CatBoost, LightGBM, XGBoost 세 모델의 OOF 예측값을 기준으로 3-model weighted ensemble을 탐색한다.
- ROC-AUC가 순위 기반 지표라는 점을 활용하여 3-model Rank Average ensemble을 추가한다.
- 확률 앙상블(raw)과 순위 앙상블(rank)을 다시 섞는 hybrid ensemble을 만든다.
- OOF 기준으로 가장 좋은 후보를 제출 파일로 저장한다.

주의할 점은 모든 가중치 선택 기준이 test 점수가 아니라 **train 내부 OOF ROC-AUC**라는 점이다.


In [ ]:
# 03-13. 개선안 실행 전 현재 사용 가능한 OOF 예측값 확인

required_oof_objects = {
    "cat_oof_pred": "CatBoost 5-Fold OOF 예측값",
    "cat_test_pred": "CatBoost 5-Fold 제출용 예측값",
    "lgb_oof_pred": "LightGBM 5-Fold OOF 예측값",
    "lgb_test_pred": "LightGBM 5-Fold 제출용 예측값",
}

for obj_name, description in required_oof_objects.items():
    if obj_name not in globals():
        raise NameError(f"{obj_name}이 없습니다. 먼저 해당 모델의 5-Fold 셀을 실행하세요. ({description})")

print("기존 OOF 예측값 확인 완료")
print("cat_oof_pred:", cat_oof_pred.shape)
print("lgb_oof_pred:", lgb_oof_pred.shape)
print("cat_test_pred:", cat_test_pred.shape)
print("lgb_test_pred:", lgb_test_pred.shape)


### 03-14. XGBoost 5-Fold OOF 추가

기존 v3에서는 XGBoost를 단일 train/valid split에서만 확인했다.  
CatBoost, LightGBM과 같은 조건으로 비교하려면 XGBoost도 5-Fold OOF 예측값을 만들어야 한다.

XGBoost 단독 점수가 CatBoost보다 낮더라도, 다른 모델과 오류 패턴이 다르면 ensemble에서 AUC 개선에 기여할 수 있다.


In [ ]:
# XGBoost 5-Fold OOF 학습
from xgboost import XGBClassifier

xgb_oof_pred = np.zeros(len(X_ordinal))
xgb_test_pred = np.zeros(len(X_test_ordinal))
xgb_fold_results = []
xgb_models = []

neg_count = (y == 0).sum()
pos_count = (y == 1).sum()
scale_pos_weight = neg_count / pos_count

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_ordinal, y), start=1):
    print(f"\n========== Fold {fold} ==========")

    X_tr = X_ordinal.iloc[train_idx]
    X_val = X_ordinal.iloc[valid_idx]
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[valid_idx]

    xgb_model = XGBClassifier(
        n_estimators=3000,
        learning_rate=0.025,
        max_depth=4,
        min_child_weight=12,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=8.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
        early_stopping_rounds=200,
    )

    xgb_model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        verbose=200,
    )

    val_pred = xgb_model.predict_proba(X_val)[:, 1]
    test_pred = xgb_model.predict_proba(X_test_ordinal)[:, 1]

    xgb_oof_pred[valid_idx] = val_pred
    xgb_test_pred += test_pred / skf.n_splits

    fold_auc = roc_auc_score(y_val, val_pred)
    fold_logloss = log_loss(y_val, val_pred)

    xgb_fold_results.append({
        "fold": fold,
        "ROC-AUC": fold_auc,
        "LogLoss": fold_logloss,
        "best_iteration": getattr(xgb_model, "best_iteration", None),
    })

    xgb_models.append(xgb_model)

    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Fold {fold} LogLoss: {fold_logloss:.6f}")

xgb_fold_results_df = pd.DataFrame(xgb_fold_results)
display(xgb_fold_results_df)

xgb_oof_metrics = evaluate_binary_model("XGBoost 5-Fold OOF", y, xgb_oof_pred)
display(pd.DataFrame([xgb_oof_metrics]).set_index("model"))

#### 03-14. 결과 해석 기준

XGBoost OOF ROC-AUC는 0.740229로 CatBoost 0.740262와 거의 비슷하다.
LightGBM 0.739595보다는 높다.

Fold별 ROC-AUC는 0.738300 ~ 0.743208 범위로, CatBoost와 비슷한 수준의 안정성을 보인다.
Fold 2가 0.743208로 가장 높고, Fold 4가 0.738300으로 가장 낮다.

LogLoss는 0.586157로 CatBoost 0.585947보다 약간 높지만 큰 차이는 아니다.
Recall도 0.784125로 CatBoost 0.784094와 거의 동일하다.

### 03-15. 3-Model OOF Weighted Ensemble

CatBoost, LightGBM, XGBoost 세 모델의 OOF 예측값을 가중 평균한다.  
가중치 탐색은 OOF ROC-AUC 기준으로 수행하며, test 예측값은 아직 사용하지 않는다.


In [ ]:
# CatBoost + LightGBM + XGBoost 3-model 가중치 탐색
three_model_results = []

for cat_weight in np.arange(0.40, 0.91, 0.05):
    for lgb_weight in np.arange(0.00, 0.61, 0.05):
        xgb_weight = 1 - cat_weight - lgb_weight

        if xgb_weight < 0:
            continue

        ensemble_oof_pred = (
            cat_weight * cat_oof_pred
            + lgb_weight * lgb_oof_pred
            + xgb_weight * xgb_oof_pred
        )

        metrics = evaluate_binary_model(
            f"Ensemble C{cat_weight:.2f}_L{lgb_weight:.2f}_X{xgb_weight:.2f}",
            y,
            ensemble_oof_pred
        )

        metrics["cat_weight"] = cat_weight
        metrics["lgb_weight"] = lgb_weight
        metrics["xgb_weight"] = xgb_weight

        three_model_results.append(metrics)

three_model_results_df = (
    pd.DataFrame(three_model_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(three_model_results_df.head(20))


In [ ]:
# 3-model weighted ensemble 제출용 예측값 생성
best_three = three_model_results_df.iloc[0]

best_cat_weight = best_three["cat_weight"]
best_lgb_weight = best_three["lgb_weight"]
best_xgb_weight = best_three["xgb_weight"]

three_model_oof_pred = (
    best_cat_weight * cat_oof_pred
    + best_lgb_weight * lgb_oof_pred
    + best_xgb_weight * xgb_oof_pred
)

three_model_test_pred = (
    best_cat_weight * cat_test_pred
    + best_lgb_weight * lgb_test_pred
    + best_xgb_weight * xgb_test_pred
)

print("Best 3-model weighted ensemble")
print(best_three[["model", "ROC-AUC", "LogLoss", "cat_weight", "lgb_weight", "xgb_weight"]])
print("OOF ROC-AUC:", roc_auc_score(y, three_model_oof_pred))

### 03-16. 3-Model Rank Average Ensemble

ROC-AUC는 예측 확률의 절대값보다 순위를 중요하게 보는 지표이다.  
따라서 각 모델의 예측값을 순위로 변환한 뒤 평균내는 Rank Average를 추가로 확인한다.

Rank Average는 LogLoss에는 불리할 수 있지만, ROC-AUC 기준에서는 제출 점수를 미세하게 올리는 후보가 될 수 있다.


In [ ]:
# 3-model Rank Average 가중치 탐색
from scipy.stats import rankdata

cat_oof_rank = rankdata(cat_oof_pred) / len(cat_oof_pred)
lgb_oof_rank = rankdata(lgb_oof_pred) / len(lgb_oof_pred)
xgb_oof_rank = rankdata(xgb_oof_pred) / len(xgb_oof_pred)

cat_test_rank = rankdata(cat_test_pred) / len(cat_test_pred)
lgb_test_rank = rankdata(lgb_test_pred) / len(lgb_test_pred)
xgb_test_rank = rankdata(xgb_test_pred) / len(xgb_test_pred)

rank_three_results = []

for cat_weight in np.arange(0.40, 0.91, 0.05):
    for lgb_weight in np.arange(0.00, 0.61, 0.05):
        xgb_weight = 1 - cat_weight - lgb_weight

        if xgb_weight < 0:
            continue

        rank_oof_pred = (
            cat_weight * cat_oof_rank
            + lgb_weight * lgb_oof_rank
            + xgb_weight * xgb_oof_rank
        )

        metrics = evaluate_binary_model(
            f"Rank Ensemble C{cat_weight:.2f}_L{lgb_weight:.2f}_X{xgb_weight:.2f}",
            y,
            rank_oof_pred
        )

        metrics["cat_weight"] = cat_weight
        metrics["lgb_weight"] = lgb_weight
        metrics["xgb_weight"] = xgb_weight

        rank_three_results.append(metrics)

rank_three_results_df = (
    pd.DataFrame(rank_three_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(rank_three_results_df.head(20))


Rank Average가 weighted ensemble보다 OOF 기준 0.000011 높다.
차이는 매우 작아서 “확실히 더 좋다”기보다는 제출 후보 하나가 추가된 정도로 보는 게 맞다.
Rank 방식은 ROC-AUC에는 유리할 수 있지만, LogLoss가 0.717259로 크게 나빠진다.
대회 평가지표가 ROC-AUC이므로 Rank 후보도 의미는 있다.
다만 이전에 3모델 rank 계열 제출이 LB에서 낮아진 경험이 있으므로, Rank 단독 제출은 신중하게 보는 게 좋다.

In [ ]:
# 3-model Rank Average 제출용 예측값 생성
best_rank_three = rank_three_results_df.iloc[0]

rank_cat_weight = best_rank_three["cat_weight"]
rank_lgb_weight = best_rank_three["lgb_weight"]
rank_xgb_weight = best_rank_three["xgb_weight"]

rank_three_oof_pred = (
    rank_cat_weight * cat_oof_rank
    + rank_lgb_weight * lgb_oof_rank
    + rank_xgb_weight * xgb_oof_rank
)

rank_three_test_pred = (
    rank_cat_weight * cat_test_rank
    + rank_lgb_weight * lgb_test_rank
    + rank_xgb_weight * xgb_test_rank
)

print("Best 3-model rank ensemble")
print(best_rank_three[["model", "ROC-AUC", "LogLoss", "cat_weight", "lgb_weight", "xgb_weight"]])
print("OOF ROC-AUC:", roc_auc_score(y, rank_three_oof_pred))

#### 03-16. 결과 해석

3-model Rank Average에서 가장 높은 OOF ROC-AUC는 `0.740559`로 나타났다.

최고 조합은 `CatBoost 0.50 + LightGBM 0.10 + XGBoost 0.40`이다.  
이는 3-model weighted ensemble의 `0.740548`보다 약 `0.000011` 높다.

Rank Average는 예측 확률값을 직접 평균하는 방식이 아니라, 각 모델의 예측 순위를 평균하는 방식이다.  
따라서 ROC-AUC처럼 순위 기반 평가지표에는 유리할 수 있지만, 확률 보정 품질을 보는 LogLoss는 나빠질 수 있다.

실제로 LogLoss는 `0.717259`로 weighted ensemble보다 높게 나타났다.  
하지만 본 대회의 핵심 평가지표는 ROC-AUC이므로, Rank Average는 제출 후보로 고려할 수 있다.

다만 이전 제출 경험상 Rank 방식은 public test에서 흔들릴 수 있으므로, 최종 제출 전에는 weighted ensemble과 hybrid ensemble을 함께 비교한다.

### 03-17. Raw + Rank Hybrid Ensemble

확률 기반 weighted ensemble은 확률 품질이 좋고, Rank Average는 ROC-AUC의 순위 특성에 잘 맞는다.  
두 예측값을 다시 섞어 OOF 기준 최적 비율을 찾는다.


In [ ]:
# Raw weighted ensemble과 Rank Average ensemble을 다시 조합
hybrid_results = []

for raw_weight in np.arange(0.0, 1.01, 0.05):
    rank_weight = 1 - raw_weight

    hybrid_oof_pred = (
        raw_weight * three_model_oof_pred
        + rank_weight * rank_three_oof_pred
    )

    metrics = evaluate_binary_model(
        f"Hybrid Raw{raw_weight:.2f}_Rank{rank_weight:.2f}",
        y,
        hybrid_oof_pred
    )

    metrics["raw_weight"] = raw_weight
    metrics["rank_weight"] = rank_weight

    hybrid_results.append(metrics)

hybrid_results_df = (
    pd.DataFrame(hybrid_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(hybrid_results_df)

In [ ]:
# Hybrid ensemble 제출용 예측값 생성
best_hybrid = hybrid_results_df.iloc[0]

best_raw_weight = best_hybrid["raw_weight"]
best_rank_weight = best_hybrid["rank_weight"]

hybrid_oof_pred = (
    best_raw_weight * three_model_oof_pred
    + best_rank_weight * rank_three_oof_pred
)

hybrid_test_pred = (
    best_raw_weight * three_model_test_pred
    + best_rank_weight * rank_three_test_pred
)

print("Best hybrid ensemble")
print(best_hybrid[["model", "ROC-AUC", "LogLoss", "raw_weight", "rank_weight"]])
print("OOF ROC-AUC:", roc_auc_score(y, hybrid_oof_pred))

#### 03-17. 결과 해석

Raw weighted ensemble과 Rank Average ensemble을 다시 조합한 결과, 가장 높은 OOF ROC-AUC는 `0.740559`로 나타났다.

최고 조합은 `Raw 0.00 + Rank 1.00`이다.  
이는 hybrid 방식에서 확률 weighted ensemble을 추가로 섞는 것보다, Rank Average 단독이 OOF 기준으로 더 적합했음을 의미한다.

다만 Rank Average는 예측값을 순위 기반으로 변환하기 때문에 LogLoss는 `0.717259`로 높게 나타났다.  
본 대회의 평가지표는 ROC-AUC이므로 OOF 기준으로는 Rank Average가 최종 후보가 될 수 있다.

하지만 이전 제출에서 Rank 기반 조합이 public leaderboard에서 낮게 나온 경험이 있으므로, 최종 제출 후보는 다음 두 가지로 나누어 판단한다.

- 공격적 후보: `3-model Rank Average C0.50_L0.10_X0.40`
- 안정적 후보: `3-model Weighted Ensemble C0.50_L0.10_X0.40`

OOF만 보면 Rank Average가 가장 높지만, 제출 안정성까지 고려하면 weighted ensemble도 함께 비교할 필요가 있다.

### 03-18. 3-Model Weighted Fine Search

기존 3-model weighted ensemble의 최고 조합은 `CatBoost 0.50 + LightGBM 0.10 + XGBoost 0.40`이었다.  
해당 탐색은 0.05 단위로 수행되었으므로, 최고 조합 주변을 0.01 단위로 세밀하게 다시 탐색한다.

이 단계에서는 모델을 다시 학습하지 않고, 이미 생성된 OOF 예측값을 조합한다.

In [ ]:
# 3-model weighted fine search

fine_three_results = []

for cat_weight in np.arange(0.42, 0.59, 0.01):
    for lgb_weight in np.arange(0.00, 0.21, 0.01):
        xgb_weight = 1 - cat_weight - lgb_weight

        if xgb_weight < 0.25 or xgb_weight > 0.55:
            continue

        fine_oof = (
            cat_weight * cat_oof_pred
            + lgb_weight * lgb_oof_pred
            + xgb_weight * xgb_oof_pred
        )

        metrics = evaluate_binary_model(
            f"Fine3 C{cat_weight:.2f}_L{lgb_weight:.2f}_X{xgb_weight:.2f}",
            y,
            fine_oof
        )

        metrics["cat_weight"] = cat_weight
        metrics["lgb_weight"] = lgb_weight
        metrics["xgb_weight"] = xgb_weight

        fine_three_results.append(metrics)

fine_three_results_df = (
    pd.DataFrame(fine_three_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(fine_three_results_df.head(20))

#### 03-18. 결과 해석

3-model weighted ensemble의 최고 조합 주변을 0.01 단위로 세밀하게 탐색하였다.

Fine search 결과, 가장 높은 OOF ROC-AUC는 약 `0.740549`로 나타났다.  
기존 조합인 `CatBoost 0.50 + LightGBM 0.10 + XGBoost 0.40`의 OOF ROC-AUC `0.740548`과 거의 동일한 수준이다.

최고 조합은 `CatBoost 0.47 + LightGBM 0.11 + XGBoost 0.42`였지만, 기존 조합 대비 개선폭은 약 `0.000001` 수준으로 매우 작다.

따라서 fine search를 통해 3-model weighted ensemble의 최적 구간이 `CatBoost 0.46~0.50`, `LightGBM 0.09~0.13`, `XGBoost 0.39~0.44` 부근이라는 점은 확인할 수 있었다.  
다만 제출 안정성과 해석 가능성을 고려하면, 기존의 단순한 `CatBoost 0.50 + LightGBM 0.10 + XGBoost 0.40` 조합을 유지하는 것이 더 적절하다.

### 03-19. v4 방식 Multi-Seed 후보 모델 추가

v4의 핵심 아이디어 중 하나는 단일 seed 모델에 의존하지 않고, 서로 다른 random seed로 여러 모델을 학습한 뒤 OOF 기준으로 앙상블하는 것이다.

동일한 모델 구조라도 seed가 달라지면 K-Fold 분할, 내부 샘플링, tree 생성 과정에서 미세하게 다른 예측 패턴이 만들어질 수 있다.  
이러한 예측 다양성은 ensemble에서 성능 안정화와 AUC 개선에 기여할 수 있다.

본 단계에서는 v5의 기존 feature set을 유지하면서, 먼저 CatBoost를 추가 seed로 학습하여 multi-seed 후보를 확장한다.

In [ ]:
# 기존 seed42 OOF/test 예측값을 multi-seed 앙상블 후보로 등록

oof_preds_v5 = {
    "cat_seed42": cat_oof_pred,
    "lgb_seed42": lgb_oof_pred,
    "xgb_seed42": xgb_oof_pred,
}

test_preds_v5 = {
    "cat_seed42": cat_test_pred,
    "lgb_seed42": lgb_test_pred,
    "xgb_seed42": xgb_test_pred,
}

score_rows_v5 = []

for name, pred in oof_preds_v5.items():
    score_rows_v5.append({
        "run_name": name,
        "model_type": name.split("_")[0],
        "seed": 42,
        "OOF ROC-AUC": roc_auc_score(y, pred),
    })

score_df_v5 = (
    pd.DataFrame(score_rows_v5)
    .sort_values("OOF ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(score_df_v5)

In [ ]:
# CatBoost 추가 seed 5-Fold OOF 학습

extra_cat_seeds = [2026, 777]

cat_extra_fold_rows = []

for seed in extra_cat_seeds:
    print(f"\n==============================")
    print(f"CatBoost extra seed: {seed}")
    print(f"==============================")

    cat_extra_oof = np.zeros(len(X_cat))
    cat_extra_test = np.zeros(len(X_test_cat))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cat, y), 1):
        print(f"\n========== Fold {fold} ==========")

        X_tr = X_cat.iloc[tr_idx].copy()
        X_val = X_cat.iloc[val_idx].copy()
        y_tr = y.iloc[tr_idx].copy()
        y_val = y.iloc[val_idx].copy()

        train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
        valid_pool = Pool(X_val, y_val, cat_features=cat_features)
        test_pool = Pool(X_test_cat, cat_features=cat_features)

        model = CatBoostClassifier(
            iterations=3000,
            learning_rate=0.03,
            depth=6,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed + fold,
            auto_class_weights="Balanced",
            early_stopping_rounds=200,
            verbose=200,
        )

        model.fit(
            train_pool,
            eval_set=valid_pool,
            use_best_model=True,
        )

        val_pred = model.predict_proba(valid_pool)[:, 1]
        test_pred = model.predict_proba(test_pool)[:, 1]

        cat_extra_oof[val_idx] = val_pred
        cat_extra_test += test_pred / skf.n_splits

        fold_auc = roc_auc_score(y_val, val_pred)
        fold_logloss = log_loss(y_val, val_pred)

        cat_extra_fold_rows.append({
            "run_name": f"cat_seed{seed}",
            "seed": seed,
            "fold": fold,
            "ROC-AUC": fold_auc,
            "LogLoss": fold_logloss,
            "best_iteration": model.get_best_iteration(),
        })

        print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")
        print(f"Fold {fold} LogLoss: {fold_logloss:.6f}")

    run_name = f"cat_seed{seed}"

    oof_preds_v5[run_name] = cat_extra_oof
    test_preds_v5[run_name] = cat_extra_test

    seed_auc = roc_auc_score(y, cat_extra_oof)

    score_rows_v5.append({
        "run_name": run_name,
        "model_type": "cat",
        "seed": seed,
        "OOF ROC-AUC": seed_auc,
    })

    print(f"\n{run_name} OOF ROC-AUC: {seed_auc:.6f}")

In [ ]:
# Multi-seed 후보 모델 성능 정리

score_df_v5 = (
    pd.DataFrame(score_rows_v5)
    .drop_duplicates(subset=["run_name"], keep="last")
    .sort_values("OOF ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

cat_extra_fold_results_df = pd.DataFrame(cat_extra_fold_rows)

display(score_df_v5)
display(cat_extra_fold_results_df)

In [ ]:
display(score_df_v5)
display(cat_extra_fold_results_df)

#### 03-19. 결과 해석

v4 방식의 multi-seed 전략을 v5에 적용하기 위해 CatBoost 모델을 서로 다른 seed로 추가 학습하였다.

기존 `cat_seed42`의 OOF ROC-AUC는 `0.740262`였고, 추가로 학습한 CatBoost 모델의 성능은 다음과 같다.

- `cat_seed777`: `0.740291`
- `cat_seed2026`: `0.740274`
- `cat_seed42`: `0.740262`

추가 seed CatBoost 모델들은 기존 seed42 모델보다 소폭 높은 OOF ROC-AUC를 보였다.  
특히 `cat_seed777`이 `0.740291`로 가장 높게 나타났다.

다만 세 CatBoost 모델 간 성능 차이는 매우 작다.  
이는 모델 구조와 feature set이 동일하기 때문에 기본적인 예측 패턴은 유사하지만, seed 차이에 따른 K-Fold 분할 및 내부 학습 과정의 미세한 차이가 존재한다는 의미이다.

Fold별 결과를 보면 `cat_seed2026`과 `cat_seed777` 모두 Fold 2에서 가장 높은 성능을 보이고, Fold 1과 Fold 4에서 상대적으로 낮은 성능을 보였다.  
따라서 성능 차이는 seed 자체보다 fold 데이터 분포의 영향을 더 크게 받는 것으로 해석된다.

그럼에도 불구하고 추가 seed 모델을 사용하는 이유는 단일 모델 성능 향상만이 아니다.  
서로 다른 seed로 학습된 CatBoost 모델은 유사하지만 완전히 동일하지 않은 예측값을 생성하므로, 이후 ensemble 단계에서 예측 안정성과 다양성을 높일 수 있다.

전체 후보 모델을 비교하면 다음과 같다.

- `cat_seed777`: `0.740291`
- `cat_seed2026`: `0.740274`
- `cat_seed42`: `0.740262`
- `xgb_seed42`: `0.740229`
- `lgb_seed42`: `0.739595`

현재 단일 모델 기준으로는 CatBoost 계열이 가장 안정적인 성능을 보인다.  
XGBoost는 CatBoost와 거의 비슷한 수준의 OOF ROC-AUC를 보이며, LightGBM은 단독 성능은 낮지만 ensemble 다양성 측면에서 활용 가치가 있다.

따라서 다음 단계에서는 `cat_seed42`, `cat_seed2026`, `cat_seed777`, `xgb_seed42`, `lgb_seed42`를 함께 사용하여 multi-seed OOF ensemble을 탐색한다.

### 03-20. Multi-Seed OOF 앙상블 탐색

03-19에서 생성한 multi-seed CatBoost 후보와 기존 LightGBM, XGBoost 후보를 함께 사용하여 ensemble을 탐색한다.

이 단계에서는 두 가지 방식을 비교한다.

1. Weighted ensemble  
   - 각 모델의 예측 확률을 가중 평균한다.

2. Rank average ensemble  
   - 각 모델의 예측값을 순위로 변환한 뒤 가중 평균한다.

탐색 기준은 test 점수가 아니라 train 내부 OOF ROC-AUC이다.

In [ ]:
# Multi-seed weighted ensemble 탐색

score_df_v5 = (
    pd.DataFrame(score_rows_v5)
    .drop_duplicates(subset=["run_name"], keep="last")
    .sort_values("OOF ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(score_df_v5)

model_names = score_df_v5["run_name"].tolist()

multi_seed_results = []

rng = np.random.default_rng(42)

for selected_count in range(2, min(6, len(model_names)) + 1):
    selected_models = model_names[:selected_count]

    # 기본 equal weight도 포함
    equal_weights = np.ones(selected_count) / selected_count

    candidate_weights = [equal_weights]

    # random dirichlet weights
    random_weights = rng.dirichlet(
        np.ones(selected_count),
        size=500
    )

    candidate_weights.extend(random_weights)

    for weights in candidate_weights:
        oof_matrix = np.column_stack([
            oof_preds_v5[name] for name in selected_models
        ])

        blend_oof = oof_matrix @ weights

        multi_seed_results.append({
            "ensemble_type": "weighted",
            "members": "|".join(selected_models),
            "weights": "|".join(f"{w:.6f}" for w in weights),
            "ROC-AUC": roc_auc_score(y, blend_oof),
        })

multi_seed_results_df = (
    pd.DataFrame(multi_seed_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(multi_seed_results_df.head(20))

In [ ]:
# Multi-seed rank average ensemble 탐색

multi_seed_rank_results = []

rng = np.random.default_rng(2026)

for selected_count in range(2, min(6, len(model_names)) + 1):
    selected_models = model_names[:selected_count]

    rank_matrix = np.column_stack([
        rankdata(oof_preds_v5[name]) / len(y)
        for name in selected_models
    ])

    equal_weights = np.ones(selected_count) / selected_count
    candidate_weights = [equal_weights]

    random_weights = rng.dirichlet(
        np.ones(selected_count),
        size=500
    )

    candidate_weights.extend(random_weights)

    for weights in candidate_weights:
        blend_rank_oof = rank_matrix @ weights

        multi_seed_rank_results.append({
            "ensemble_type": "rank",
            "members": "|".join(selected_models),
            "weights": "|".join(f"{w:.6f}" for w in weights),
            "ROC-AUC": roc_auc_score(y, blend_rank_oof),
        })

multi_seed_rank_results_df = (
    pd.DataFrame(multi_seed_rank_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(multi_seed_rank_results_df.head(20))

In [ ]:
display(multi_seed_results_df.head(20))
display(multi_seed_rank_results_df.head(20))

## 04. 평가

모델별 성능을 ROC-AUC, LogLoss, Accuracy, Precision, Recall, F1-score 기준으로 비교한다.  
본 프로젝트의 핵심 평가지표는 ROC-AUC이며, 클래스 불균형을 고려하여 Recall과 F1-score도 함께 확인한다.

### 04-1. 평가 지표 정리

본 프로젝트에서 사용하는 주요 평가 지표의 의미는 다음과 같다.

1. ROC-AUC
- 임신 성공/실패를 얼마나 잘 구분하는지 평가하는 지표이다.
- 예측 확률의 순위가 실제 정답 순서와 잘 맞을수록 값이 높아진다.
- 본 프로젝트의 핵심 평가 지표이다.

2. LogLoss
- 예측 확률이 실제 정답과 얼마나 가까운지 평가하는 지표이다.
- 낮을수록 좋은 성능을 의미한다.
- 확신 있게 틀린 예측에 큰 패널티를 준다.

3. Accuracy
- 전체 데이터 중 맞게 예측한 비율이다.
- 클래스 불균형이 있을 경우 성능을 과대평가할 수 있어 보조 지표로만 활용한다.

4. Precision
- 모델이 임신 성공으로 예측한 사례 중 실제 성공한 비율이다.
- 성공 예측의 정확도를 나타낸다.

5. Recall
- 실제 임신 성공 사례 중 모델이 성공으로 잘 잡아낸 비율이다.
- 성공 클래스를 놓치지 않는 정도를 나타낸다.

6. F1-score
- Precision과 Recall의 조화평균이다.
- 불균형 데이터에서 성공 클래스 예측 성능을 함께 보기 위한 보조 지표이다.

### 04-2. 모델별 최종 성능 비교

03 모델링 단계에서 학습한 모델들의 단독 검증 성능을 비교한다.

In [ ]:
# 모델별 성능 비교표 정리
final_model_results = model_compare_with_ensemble.copy()
final_model_results = final_model_results.sort_values("ROC-AUC", ascending=False)

display(final_model_results)

## 05. 제출 및 리더보드 확인

최종 모델 또는 앙상블 모델로 test 데이터의 임신 성공 확률을 예측하고, 제출 형식에 맞는 CSV 파일을 생성한다.


In [ ]:
#5월 28일차 단일 캣부스트가 무려 1일차 최상위 점수보다 높게 나왔습니다! 내일 앙상블 포함 여러 시도 해보겠습니다!
### 05-1. CatBoost 5-Fold 제출 파일 생성

# 제출 파일 기본 형식 복사
submission_catboost_5fold = submission.copy()

# 예측 확률 컬럼명 자동 확인
pred_col = [col for col in submission_catboost_5fold.columns if col != "ID"][0]

# CatBoost 5-Fold test 예측값 입력
submission_catboost_5fold[pred_col] = cat_test_pred

# 제출 파일 저장
submission_path = OUTPUT_DIR / "submission_catboost_5fold_oof.csv"
submission_catboost_5fold.to_csv(submission_path, index=False)

print("저장 완료:", submission_path)
display(submission_catboost_5fold.head())

In [ ]:
### 05-2. 제출 파일 검증

submission_check = pd.read_csv(submission_path)

print("shape:", submission_check.shape)
print("columns:", submission_check.columns.tolist())
print("ID 일치:", (submission_check["ID"] == submission["ID"]).all())
print("결측값 수:", submission_check.isna().sum().sum())
print("확률 범위:", submission_check[pred_col].min(), "~", submission_check[pred_col].max())

display(submission_check[pred_col].describe())
display(submission_check.head())

In [ ]:
### 05-3. CatBoost + LightGBM 앙상블 제출 파일 생성

best_cat_weight = 0.70
best_lgb_weight = 0.30

ensemble_test_pred = (
    best_cat_weight * cat_test_pred +
    best_lgb_weight * lgb_test_pred
)

ensemble_test_pred = np.clip(ensemble_test_pred, 0, 1)

submission_cat_lgb = submission.copy()
pred_col = [col for col in submission_cat_lgb.columns if col != "ID"][0]

submission_cat_lgb[pred_col] = ensemble_test_pred

submission_path = OUTPUT_DIR / "submission_cat_lgb_oof_C070_L030.csv"
submission_cat_lgb.to_csv(submission_path, index=False)

print("저장 완료:", submission_path)
display(submission_cat_lgb.head())

In [ ]:
### 05-4. 앙상블 제출 파일 검증

submission_check = pd.read_csv(submission_path)

print("shape:", submission_check.shape)
print("columns:", submission_check.columns.tolist())
print("ID 일치:", (submission_check["ID"] == submission["ID"]).all())
print("결측값 수:", submission_check.isna().sum().sum())
print("확률 범위:", submission_check[pred_col].min(), "~", submission_check[pred_col].max())

display(submission_check[pred_col].describe())
display(submission_check.head())

In [ ]:
### 05-5. Rank Average 앙상블 제출 파일 생성

from scipy.stats import rankdata

best_cat_weight = 0.69
best_lgb_weight = 0.31

cat_test_rank = rankdata(cat_test_pred) / len(cat_test_pred)
lgb_test_rank = rankdata(lgb_test_pred) / len(lgb_test_pred)

rank_ensemble_test_pred = (
    best_cat_weight * cat_test_rank +
    best_lgb_weight * lgb_test_rank
)

submission_rank_cat_lgb = submission.copy()
pred_col = [col for col in submission_rank_cat_lgb.columns if col != "ID"][0]

submission_rank_cat_lgb[pred_col] = rank_ensemble_test_pred

submission_path = OUTPUT_DIR / "submission_rank_cat_lgb_C069_L031.csv"
submission_rank_cat_lgb.to_csv(submission_path, index=False)

print("저장 완료:", submission_path)
display(submission_rank_cat_lgb.head())

In [ ]:
### 05-6. Rank Average 제출 파일 검증

submission_check = pd.read_csv(submission_path)

print("shape:", submission_check.shape)
print("columns:", submission_check.columns.tolist())
print("ID 일치:", (submission_check["ID"] == submission["ID"]).all())
print("결측값 수:", submission_check.isna().sum().sum())
print("확률 범위:", submission_check[pred_col].min(), "~", submission_check[pred_col].max())

display(submission_check[pred_col].describe())
display(submission_check.head())

### 05-7. Multi-Seed Weighted Ensemble 제출 파일 생성

03-20에서 OOF ROC-AUC 기준 가장 안정적인 후보로 판단한 multi-seed weighted ensemble을 제출 파일로 생성한다.

이 조합은 `cat_seed777`, `cat_seed2026`, `cat_seed42`, `xgb_seed42`, `lgb_seed42`를 함께 사용하며, OOF 기준으로 기존 3-model ensemble보다 소폭 개선되었다.

In [ ]:
# Multi-seed weighted ensemble 제출 파일 생성

selected_multi_seed = multi_seed_results_df.iloc[0]

selected_names = selected_multi_seed["members"].split("|")
selected_weights = np.array([
    float(x) for x in selected_multi_seed["weights"].split("|")
])

selected_weights = selected_weights / selected_weights.sum()

multi_seed_test_matrix = np.column_stack([
    test_preds_v5[name] for name in selected_names
])

multi_seed_weighted_test_pred = multi_seed_test_matrix @ selected_weights
multi_seed_weighted_test_pred = np.clip(multi_seed_weighted_test_pred, 0, 1)

pred_col = [col for col in submission.columns if col != ID_COL][0]

submission_multi_seed_weighted = submission.copy()
submission_multi_seed_weighted[pred_col] = multi_seed_weighted_test_pred

multi_seed_weighted_path = OUTPUT_DIR / "submission_multi_seed_weighted.csv"
submission_multi_seed_weighted.to_csv(multi_seed_weighted_path, index=False)

print("저장 완료:", multi_seed_weighted_path)
print("선택 조합:", selected_multi_seed["members"])
print("선택 가중치:", selected_multi_seed["weights"])
print("OOF ROC-AUC:", selected_multi_seed["ROC-AUC"])

display(submission_multi_seed_weighted.head())